# Clinical Synthetic Data Generation Framework

This notebook explores the performance of the following Synthetic Table Generation Methods

- **CTGAN** (Conditional Tabular GAN)
- **CTAB-GAN** (Conditional Tabular GAN with advanced preprocessing)
- **CTAB-GAN+** (Enhanced version with WGAN-GP losses, general transforms, and improved stability)
- **GANerAid** (Custom implementation)
- **CopulaGAN** (Copula-based GAN)
- **TVAE** (Variational Autoencoder)

- Section 1 prepares the environment and sources setup.py.
- Section 2 reads in the dataset and produces an initial suite of EDA. 
  - 2.1.1 - user needs to adapt for incoming dataset
  - 2.1.2 - user should review to ensure target colummns and categorical columns are properly identified
  - 2.1.3 - user should confirm that data has loaded properly
  - 2.1.4 - if your data has missing values, MICE is employed; user should review
  - CHUNK_2_1_4_C: This code samples 500 rows to be used in rest of notebook for purposes of testing; comment out this code for production.
  - 2.1.5 - Exploratory data analysis
- Section 3 demonstrates the performance of each metholodogy with some default hyperparameters. 
   - 3.1.1-3.1.6 Subsections for each method
   - 3.2 batch processing of figures/tables from section 3 
- Section 4 runs hyperparameter optimization. 
   - 4.1.1-4.1.6 Subsections for each method
   - 4.2 batch processing of figures/tables from section 4 describing the optimization process 
- Section 5 re-runs each model with their respective best hyperparameters. 
   - 5.1.1-5.1.6 re-run each model using the best hyperparameters identified in Section 4.1.1-4.1.6, resp.
   - 5.2 batch processing of figures/tables from Section 5.


Refer to readme.md, doc\Model-descriptions.md, doc\Objective-function.md.

## 1 Setup and Configuration

In [1]:
# !pip install -r requirements.txt
# !pip install sdv
# !pip install ctgan
# !pip install numpy==1.26.4
# !pip install GANerAid

In [2]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
from setup import *

print("🎯 SETUP MODULE IMPORTED SUCCESSFULLY!")
print("="*60)

[OK] Essential data science libraries imported successfully!


WARNING	src.models.implementations.copulagan_model:copulagan_model.py:<module>()- Could not import preprocessing functions: cannot import name 'ModelFactory' from partially initialized module 'src.models.model_factory' (most likely due to a circular import) (c:\Users\gcicc\claudeproj\tableGenCompare\src\models\model_factory.py)


[CONFIG] Session timestamp: 2025-12-16
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
Detected sklearn 1.7.1 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.1
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully from ./CTAB-GAN
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementations


WARNING	src.models.implementations.copulagan_model:copulagan_model.py:<module>()- Current working directory: c:\Users\gcicc\claudeproj\tableGenCompare
WARNING	src.models.implementations.copulagan_model:copulagan_model.py:<module>()- File location: c:\Users\gcicc\claudeproj\tableGenCompare\src\models\implementations\copulagan_model.py
WARNING	src.models.implementations.copulagan_model:copulagan_model.py:<module>()- Project root attempted: c:\Users\gcicc\claudeproj\tableGenCompare


[OK] Backward compatibility patches loaded from src/compat.py
[OK] TRTSEvaluator backward compatibility patch applied successfully!
[SETUP] Thin re-export layer loaded successfully!
[SETUP] Session timestamp: 2025-12-16
[SETUP] All functions imported from modular src/ structure
[SETUP] Backward compatibility maintained for notebooks
🎯 SETUP MODULE IMPORTED SUCCESSFULLY!


## 2 Data Loading and Pre-processing

#### 2.1.1 USER ATTENTION NEEDED

Adapt this for your incoming dataset.

In [3]:
# Code Chunk ID: CHUNK_2_1_1_A
# =================== USER CONFIGURATION ===================
# 📝 CONFIGURE YOUR DATASET: Update these settings for your data
DATA_FILE = 'data/liver_train.csv'            # Path to your CSV file
TARGET_COLUMN = 'Result'                       # Name of your target/outcome column

# 🔧 DATASET IDENTIFIER (for results folder naming)
# Option 1: Manual override (recommended for consistent naming)
DATASET_IDENTIFIER_OVERRIDE = 'liver-train'  # Changed to match auto-extraction pattern

# 🔧 OPTIONAL ADVANCED SETTINGS (Auto-detected if left empty)
CATEGORICAL_COLUMNS = ['Gender of the patient'] # List categorical columns or leave empty for auto-detection
MISSING_STRATEGY = 'mice'                    # Options: 'mice', 'drop', 'median', 'mode'
DATASET_NAME = 'Liver Disease Dataset'        # Descriptive name for your dataset

# 🚨 IMPORTANT: Verify these settings match your dataset before running!
print(f"📊 Configuration Summary:")
print(f"   Dataset: {DATASET_NAME}")
print(f"   File: {DATA_FILE}")
print(f"   Target: {TARGET_COLUMN}")
print(f"   Manual ID Override: {DATASET_IDENTIFIER_OVERRIDE}")
print(f"   Categorical: {CATEGORICAL_COLUMNS}")
print(f"   Missing Data Strategy: {MISSING_STRATEGY}")

# Load and prepare the dataset
data_file = DATA_FILE
target_column = TARGET_COLUMN

print(f"\n🔍 Loading dataset from: {data_file}")

try:
    # 🔧 ENCODING FIX: Try multiple encodings to handle special characters
    encoding_attempts = ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']
    data = None
    
    for encoding in encoding_attempts:
        try:
            data = pd.read_csv(data_file, encoding=encoding)
            print(f"✅ Dataset loaded successfully using {encoding} encoding!")
            break
        except UnicodeDecodeError:
            print(f"⚠️  Failed with {encoding} encoding, trying next...")
            continue
    
    if data is None:
        raise Exception("Could not load file with any supported encoding")
        
    print(f"📊 Original shape: {data.shape}")
    
    # Set up dataset identifier and current data file for new folder structure
    import setup
    if DATASET_IDENTIFIER_OVERRIDE:
        dataset_identifier = DATASET_IDENTIFIER_OVERRIDE
        setup.DATASET_IDENTIFIER = DATASET_IDENTIFIER_OVERRIDE
        setup.CURRENT_DATA_FILE = data_file
        print(f"📁 Using manual dataset identifier: {dataset_identifier}")
    else:
        dataset_identifier = setup.extract_dataset_identifier(data_file)
        setup.DATASET_IDENTIFIER = dataset_identifier
        setup.CURRENT_DATA_FILE = data_file
        print(f"📁 Auto-extracted dataset identifier: {dataset_identifier}")
    
    # 🔧 CRITICAL FIX: Set global DATASET_IDENTIFIER for use in other chunks
    DATASET_IDENTIFIER = dataset_identifier  # This was missing!
    
    # 📁 NEW: Update RESULTS_DIR for organized file outputs using proper structure
    # Don't set a specific RESULTS_DIR here - let each section use get_results_path()
    # This ensures proper date/section structure like: results/dataset/2025-09-12/Section-2/
    RESULTS_DIR = f"results/{dataset_identifier}/"  # Base directory only
    
    print(f"✅ Dataset identifier set: {dataset_identifier}")
    print(f"✅ Global DATASET_IDENTIFIER: {DATASET_IDENTIFIER}")
    print(f"📅 Session timestamp: {setup.SESSION_TIMESTAMP}")
    print(f"🗂️  Results will be saved to: results/{dataset_identifier}/")
    
except FileNotFoundError:
    print(f"❌ Error: File not found at {data_file}")
    print("   Please check the DATA_FILE path in your configuration above")
    print("   Current working directory:", os.getcwd())
    raise

except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

if data is not None:
    print(f"\n📋 Dataset Info:")
    print(f"   • Shape: {data.shape}")
    print(f"   • Columns: {list(data.columns)}")
    
    # Check if target column exists
    if target_column not in data.columns:
        print(f"\n❌ WARNING: Target column '{target_column}' not found!")
        print(f"   Available columns: {list(data.columns)}")
        print("   Please update TARGET_COLUMN in the configuration above")
    else:
        print(f"   • Target column '{target_column}' found ✅")
        print(f"   • Target distribution: {data[target_column].value_counts().to_dict()}")
    
    # Check for missing values
    missing_values = data.isnull().sum()
    if missing_values.sum() > 0:
        print(f"\n⚠️  Missing values detected:")
        for col, count in missing_values[missing_values > 0].items():
            print(f"   • {col}: {count} missing ({count/len(data)*100:.1f}%)")
    else:
        print(f"\n✅ No missing values detected")
else:
    print("\n❌ Dataset loading failed - please fix the configuration and try again")

📊 Configuration Summary:
   Dataset: Liver Disease Dataset
   File: data/liver_train.csv
   Target: Result
   Manual ID Override: liver-train
   Categorical: ['Gender of the patient']
   Missing Data Strategy: mice

🔍 Loading dataset from: data/liver_train.csv
✅ Dataset loaded successfully using utf-8 encoding!
📊 Original shape: (30691, 11)
📁 Using manual dataset identifier: liver-train
✅ Dataset identifier set: liver-train
✅ Global DATASET_IDENTIFIER: liver-train
📅 Session timestamp: 2025-12-16
🗂️  Results will be saved to: results/liver-train/

📋 Dataset Info:
   • Shape: (30691, 11)
   • Columns: ['Age of the patient', 'Gender of the patient', 'Total Bilirubin', 'Direct Bilirubin', 'Alkphos Alkaline Phosphotase', 'Sgpt Alamine Aminotransferase', 'Sgot Aspartate Aminotransferase', 'Total Protiens', 'ALB Albumin', 'A/G Ratio Albumin and Globulin Ratio', 'Result']
   • Target column 'Result' found ✅
   • Target distribution: {1: 21917, 2: 8774}

⚠️  Missing values detected:
   • Age of

The code defines utilities for column name standardization and dataset analysis using the pandas library in Python. It includes functions to clean and normalize column names, identify the target variable, categorize column types, and validate dataset configurations. These functions enhance data preprocessing by ensuring consistency and integrity, making it easier to manage various data types and handle potential issues like missing values. Overall, they provide a structured approach for effective dataset analysis.

#### 2.1.2 Column Name Standardization and Dataset Analysis Utilities

In [4]:
# Code Chunk ID: CHUNK_2_1_2_A
# Column Name Standardization and Dataset Analysis Utilities
import os
import pandas as pd
import re
import numpy as np
from typing import Dict, List, Tuple, Any
import optuna

def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Create mapping of old to new column names
    name_mapping = {}
    
    for col in df.columns:
        # Remove special characters and normalize
        new_name = re.sub(r'[^\w\s]', '', str(col))  # Remove special chars
        new_name = re.sub(r'\s+', '_', new_name.strip())  # Replace spaces with underscores
        new_name = new_name.lower()  # Convert to lowercase
        
        # Handle duplicate names
        if new_name in name_mapping.values():
            counter = 1
            while f"{new_name}_{counter}" in name_mapping.values():
                counter += 1
            new_name = f"{new_name}_{counter}"
            
        name_mapping[col] = new_name
    
    # Rename columns
    df = df.rename(columns=name_mapping)
    
    print(f"🔄 Column Name Standardization:")
    for old, new in name_mapping.items():
        if old != new:
            print(f"   '{old}' → '{new}'")
    
    return df, name_mapping

def detect_target_column(df: pd.DataFrame, target_hint: str = None) -> str:
    """
    Detect the target column in the dataset.
    
    Args:
        df: Input dataframe
        target_hint: User-provided hint for target column name
        
    Returns:
        Name of the detected target column
    """
    # Common target column patterns
    target_patterns = [
        'target', 'label', 'class', 'outcome', 'result', 'diagnosis', 
        'response', 'y', 'dependent', 'output', 'prediction'
    ]
    
    # If user provided hint, try to find it first
    if target_hint:
        # Try exact match (case insensitive)
        for col in df.columns:
            if col.lower() == target_hint.lower():
                print(f"✅ Target column found: '{col}' (user specified)")
                return col
        
        # Try partial match
        for col in df.columns:
            if target_hint.lower() in col.lower():
                print(f"✅ Target column found: '{col}' (partial match to '{target_hint}')")
                return col
    
    # Auto-detect based on patterns
    for pattern in target_patterns:
        for col in df.columns:
            if pattern in col.lower():
                print(f"✅ Target column auto-detected: '{col}' (pattern: '{pattern}')")
                return col
    
    # If no pattern match, check for binary columns (likely targets)
    binary_cols = []
    for col in df.columns:
        unique_vals = df[col].dropna().nunique()
        if unique_vals == 2:
            binary_cols.append(col)
    
    if binary_cols:
        target_col = binary_cols[0]  # Take first binary column
        print(f"✅ Target column inferred: '{target_col}' (binary column)")
        return target_col
    
    # Last resort: use last column
    target_col = df.columns[-1]
    print(f"⚠️ Target column defaulted to: '{target_col}' (last column)")
    return target_col

def analyze_column_types(df: pd.DataFrame, categorical_hint: List[str] = None) -> Dict[str, str]:
    """
    Analyze and categorize column types.
    
    Args:
        df: Input dataframe
        categorical_hint: User-provided list of categorical columns
        
    Returns:
        Dictionary mapping column names to types ('categorical', 'continuous', 'binary')
    """
    column_types = {}
    
    for col in df.columns:
        # Skip if user explicitly specified as categorical
        if categorical_hint and col in categorical_hint:
            column_types[col] = 'categorical'
            continue
            
        # Analyze column characteristics
        non_null_data = df[col].dropna()
        unique_count = non_null_data.nunique()
        total_count = len(non_null_data)
        
        # Determine type based on data characteristics
        if unique_count == 2:
            column_types[col] = 'binary'
        elif df[col].dtype == 'object' or unique_count < 10:
            column_types[col] = 'categorical'
        elif df[col].dtype in ['int64', 'float64'] and unique_count > 10:
            column_types[col] = 'continuous'
        else:
            # Default based on uniqueness ratio
            uniqueness_ratio = unique_count / total_count
            if uniqueness_ratio < 0.1:
                column_types[col] = 'categorical'
            else:
                column_types[col] = 'continuous'
    
    return column_types

def validate_dataset_config(df: pd.DataFrame, target_col: str, config: Dict[str, Any]) -> bool:
    """
    Validate dataset configuration and provide warnings.
    
    Args:
        df: Input dataframe
        target_col: Target column name
        config: Configuration dictionary
        
    Returns:
        True if validation passes, False otherwise
    """
    print(f"\n🔍 Dataset Validation:")
    
    valid = True
    
    # Check if target column exists
    if target_col not in df.columns:
        print(f"❌ Target column '{target_col}' not found in dataset!")
        print(f"   Available columns: {list(df.columns)}")
        valid = False
    else:
        print(f"✅ Target column '{target_col}' found")
    
    # Check dataset size
    if len(df) < 100:
        print(f"⚠️ Small dataset: {len(df)} rows (recommend >1000 for synthetic data)")
    else:
        print(f"✅ Dataset size: {len(df)} rows")
    
    # Check for missing data
    missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
    if missing_pct > 20:
        print(f"⚠️ High missing data: {missing_pct:.1f}% (recommend MICE imputation)")
    elif missing_pct > 0:
        print(f"🔍 Missing data: {missing_pct:.1f}% (manageable)")
    else:
        print(f"✅ No missing data")
    
    return valid

print("✅ Dataset analysis utilities loaded successfully!")

✅ Dataset analysis utilities loaded successfully!


#### 2.1.3 Load and Analyze Dataset with Generalized Configuration

This code loads and analyzes a dataset using a specified configuration. It imports necessary libraries, attempts to read a CSV file, and standardizes the column names while allowing for potential updates to the target column. The script detects the target column, analyzes data types, and validates the dataset configuration, providing a summary of the dataset’s shape and missing values. Finally, it stores metadata about the dataset for future reference.

This code provides advanced utilities for handling missing data using various strategies in Python. It includes functions to assess missing data patterns, apply Multiple Imputation by Chained Equations (MICE), visualize missing patterns, and implement different strategies for managing missing values. The `assess_missing_patterns` function analyzes and summarizes missing data, while `apply_mice_imputation` leverages an iterative imputer for numeric columns. The `visualize_missing_patterns` function creates visual representations of missing data, and the `handle_missing_data_strategy` function executes the chosen strategy, offering options like MICE, dropping rows, or filling with median or mode values. Collectively, these utilities facilitate effective management of missing data to improve dataset quality.

#### 2.1.4 Advanced Missing Data Handling with MICE

This code provides a comprehensive toolkit for analyzing, visualizing, and handling missing data in a pandas DataFrame using advanced and flexible approaches. It includes functions to assess the extent and patterns of missingness, visualize those patterns, and apply various imputation techniques. The centerpiece is the ability to perform Multiple Imputation by Chained Equations (MICE) using scikit-learn’s IterativeImputer with a RandomForestRegressor (for numerical features), which statistically estimates and fills in missing values based on all other feature relationships. For categorical variables, the code defaults to simpler mode imputation. Users can also choose alternative strategies like dropping rows (drop), filling with column medians (median), or filling with the most frequent value (mode). The code features detailed feedback and visual support with heatmaps and bar plots to help the user understand and monitor missing data patterns throughout the handling process. Together, these utilities help users robustly prepare data for machine learning or statistical analysis, reducing bias and maximizing data retention and utility.

In [5]:
# Code Chunk ID: CHUNK_2_1_4_A
# Advanced Missing Data Handling with MICE
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def assess_missing_patterns(df: pd.DataFrame) -> dict:
    """
    Comprehensive assessment of missing data patterns.
    
    Args:
        df: Input dataframe
        
    Returns:
        Dictionary with missing data analysis
    """
    missing_analysis = {}
    
    # Basic missing statistics
    missing_counts = df.isnull().sum()
    missing_percentages = (missing_counts / len(df)) * 100
    
    missing_analysis['missing_counts'] = missing_counts[missing_counts > 0]
    missing_analysis['missing_percentages'] = missing_percentages[missing_percentages > 0]
    missing_analysis['total_missing_cells'] = df.isnull().sum().sum()
    missing_analysis['total_cells'] = df.size
    missing_analysis['overall_missing_rate'] = (missing_analysis['total_missing_cells'] / missing_analysis['total_cells']) * 100
    
    # Missing patterns
    missing_patterns = df.isnull().value_counts()
    missing_analysis['missing_patterns'] = missing_patterns
    
    return missing_analysis

def apply_mice_imputation(df: pd.DataFrame, target_col: str = None, max_iter: int = 10, random_state: int = 42) -> pd.DataFrame:
    """
    Apply Multiple Imputation by Chained Equations (MICE) to handle missing data.
    
    Args:
        df: Input dataframe with missing values
        target_col: Target column name (excluded from imputation predictors)
        max_iter: Maximum number of imputation iterations
        random_state: Random state for reproducibility
        
    Returns:
        DataFrame with imputed values
    """
    print(f"🔧 Applying MICE imputation...")
    
    # Separate features and target
    if target_col and target_col in df.columns:
        features = df.drop(columns=[target_col])
        target = df[target_col]
    else:
        features = df.copy()
        target = None
    
    # Identify numeric and categorical columns
    numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()
    
    df_imputed = features.copy()
    
    # Handle numeric columns with MICE
    if numeric_cols:
        print(f"   Imputing {len(numeric_cols)} numeric columns...")
        numeric_imputer = IterativeImputer(
            estimator=RandomForestRegressor(n_estimators=10, random_state=random_state),
            max_iter=max_iter,
            random_state=random_state
        )
        
        numeric_imputed = numeric_imputer.fit_transform(features[numeric_cols])
        df_imputed[numeric_cols] = numeric_imputed
    
    # Handle categorical columns with mode imputation (simpler approach)
    if categorical_cols:
        print(f"   Imputing {len(categorical_cols)} categorical columns with mode...")
        for col in categorical_cols:
            mode_value = features[col].mode()
            if len(mode_value) > 0:
                df_imputed[col] = features[col].fillna(mode_value[0])
            else:
                # If no mode, fill with 'Unknown'
                df_imputed[col] = features[col].fillna('Unknown')
    
    # Add target back if it exists
    if target is not None:
        df_imputed[target_col] = target
    
    print(f"✅ MICE imputation completed!")
    print(f"   Missing values before: {features.isnull().sum().sum()}")
    print(f"   Missing values after: {df_imputed.isnull().sum().sum()}")
    
    return df_imputed

def visualize_missing_patterns(df: pd.DataFrame, title: str = "Missing Data Patterns") -> None:
    """
    Create visualizations for missing data patterns.
    
    Args:
        df: Input dataframe
        title: Title for the plot
    """
    missing_data = df.isnull()
    
    if missing_data.sum().sum() == 0:
        print("✅ No missing data to visualize!")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Missing data heatmap
    sns.heatmap(missing_data, 
                yticklabels=False, 
                cbar=True, 
                cmap='viridis',
                ax=axes[0])
    axes[0].set_title('Missing Data Heatmap')
    axes[0].set_xlabel('Columns')
    
    # Missing data bar chart
    missing_counts = missing_data.sum()
    missing_counts = missing_counts[missing_counts > 0]
    
    if len(missing_counts) > 0:
        missing_counts.plot(kind='bar', ax=axes[1], color='coral')
        axes[1].set_title('Missing Values by Column')
        axes[1].set_ylabel('Count of Missing Values')
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, 'No Missing Data', 
                    horizontalalignment='center', 
                    verticalalignment='center',
                    transform=axes[1].transAxes,
                    fontsize=16)
        axes[1].set_title('Missing Values by Column')
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

def handle_missing_data_strategy(df: pd.DataFrame, strategy: str, target_col: str = None) -> pd.DataFrame:
    """
    Apply the specified missing data handling strategy.
    
    Args:
        df: Input dataframe
        strategy: Strategy to use ('mice', 'drop', 'median', 'mode')
        target_col: Target column name
        
    Returns:
        DataFrame with missing data handled
    """
    print(f"\n🔧 Applying missing data strategy: {strategy.upper()}")
    
    if df.isnull().sum().sum() == 0:
        print("✅ No missing data detected - no imputation needed")
        return df.copy()
    
    if strategy.lower() == 'mice':
        return apply_mice_imputation(df, target_col)
    
    elif strategy.lower() == 'drop':
        print(f"   Dropping rows with missing values...")
        df_clean = df.dropna()
        print(f"   Rows before: {len(df)}, Rows after: {len(df_clean)}")
        return df_clean
    
    elif strategy.lower() == 'median':
        print(f"   Filling missing values with median/mode...")
        df_filled = df.copy()
        
        # Numeric columns: fill with median
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df_filled[col] = df[col].fillna(median_val)
                print(f"     {col}: filled {df[col].isnull().sum()} values with median {median_val:.2f}")
        
        # Categorical columns: fill with mode
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        for col in categorical_cols:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()
                if len(mode_val) > 0:
                    df_filled[col] = df[col].fillna(mode_val[0])
                    print(f"     {col}: filled {df[col].isnull().sum()} values with mode '{mode_val[0]}'")
        
        return df_filled
    
    elif strategy.lower() == 'mode':
        print(f"   Filling missing values with mode...")
        df_filled = df.copy()
        
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()
                if len(mode_val) > 0:
                    df_filled[col] = df[col].fillna(mode_val[0])
                    print(f"     {col}: filled {df[col].isnull().sum()} values with mode '{mode_val[0]}'")
        
        return df_filled
    
    else:
        print(f"⚠️ Unknown strategy '{strategy}'. Using 'median' as fallback.")
        return handle_missing_data_strategy(df, 'median', target_col)

print("✅ Missing data handling utilities loaded successfully!")

✅ Missing data handling utilities loaded successfully!


In [6]:
# Code Chunk ID: CHUNK_2_1_4_B
# ============================================================================
# CONDITIONAL MISSING DATA IMPUTATION
# ============================================================================
# Apply missing data strategy only if missing values exist

missing_count = data.isnull().sum().sum()

if missing_count > 0:
    print(f"🔧 MISSING DATA IMPUTATION")
    print(f"📊 Found {missing_count} missing values - applying {MISSING_STRATEGY} strategy")
    
    # Store original data
    data_original = data.copy()
    
    # Apply imputation using CHUNK_008 functions
    data = handle_missing_data_strategy(data, MISSING_STRATEGY, TARGET_COLUMN)
    
    # Report results
    remaining = data.isnull().sum().sum()
    print(f"✅ Imputation complete: {missing_count} → {remaining} missing values")
else:
    print("✅ No missing values detected - skipping imputation")

🔧 MISSING DATA IMPUTATION
📊 Found 5425 missing values - applying mice strategy

🔧 Applying missing data strategy: MICE
🔧 Applying MICE imputation...
   Imputing 9 numeric columns...
   Imputing 1 categorical columns with mode...
✅ MICE imputation completed!
   Missing values before: 5425
   Missing values after: 0
✅ Imputation complete: 5425 → 0 missing values


In [7]:
# Code Chunk ID: CHUNK_2_1_4_C
# ============================================================================
# ROBUST MISSING DATA IMPUTATION
# ============================================================================
# Apply missing data strategy with robust error handling

missing_count = data.isnull().sum().sum()

if missing_count > 0:
    print(f"🔧 ROBUST MISSING DATA IMPUTATION")
    print(f"📊 Found {missing_count} missing values - applying {MISSING_STRATEGY} strategy with error handling")
    
    try:
        # Store original data for fallback
        data_original = data.copy()
        
        # Apply imputation with error handling
        data = handle_missing_data_strategy(data, MISSING_STRATEGY, TARGET_COLUMN)
        
        # Verify imputation success
        remaining = data.isnull().sum().sum()
        
        if remaining < missing_count:
            print(f"✅ Robust imputation successful: {missing_count} → {remaining} missing values")
        else:
            print(f"⚠️  Imputation may not be complete: {missing_count} → {remaining} missing values")
            
    except Exception as e:
        print(f"❌ Error during imputation: {e}")
        print("🔄 Using fallback: median imputation for numeric, mode for categorical")
        
        # Fallback imputation strategy
        numeric_columns = data.select_dtypes(include=[np.number]).columns
        categorical_columns = data.select_dtypes(include=['object']).columns
        
        for col in numeric_columns:
            if data[col].isnull().sum() > 0:
                data[col].fillna(data[col].median(), inplace=True)
                
        for col in categorical_columns:
            if data[col].isnull().sum() > 0:
                data[col].fillna(data[col].mode()[0], inplace=True)
                
        remaining = data.isnull().sum().sum()
        print(f"✅ Fallback imputation complete: {missing_count} → {remaining} missing values")
        
else:
    print("✅ No missing values detected - skipping robust imputation")

✅ No missing values detected - skipping robust imputation


⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️

Note that next chunk samples 500 rows

⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️⚠️

In [8]:
# CHUNK_2_1_4_C
# This chunk samples 500 rows for quicker testing - remove for full dataset
data=data.sample(n=2500, random_state=42)
# data.shape
# data.head

#### 2.1.5 EDA
This code snippet provides an enhanced overview and analysis of a dataset. It generates basic statistics, including the dataset name, shape, memory usage, total missing values, missing percentage, number of duplicate rows, and counts of numeric and categorical columns. The results are organized into a dictionary called `overview_stats`, which is then iterated over to print each statistic in a formatted manner. Additionally, it sets up for displaying a sample of the data afterward.

In [9]:
# Code Chunk ID: CHUNK_2_1_5_A
# Enhanced Dataset Overview and Analysis
print("📋 COMPREHENSIVE DATASET OVERVIEW")
print("=" * 60)

# Basic statistics
overview_stats = {
    'Dataset Name': 'Breast Cancer Wisconsin (Diagnostic)',
    'Shape': f"{data.shape[0]} rows × {data.shape[1]} columns",
    'Memory Usage': f"{data.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
    'Total Missing Values': data.isnull().sum().sum(),
    'Missing Percentage': f"{(data.isnull().sum().sum() / data.size) * 100:.2f}%",
    'Duplicate Rows': data.duplicated().sum(),
    'Numeric Columns': len(data.select_dtypes(include=[np.number]).columns),
    'Categorical Columns': len(data.select_dtypes(include=['object']).columns)
}

for key, value in overview_stats.items():
    print(f"{key:.<25} {value}")

📋 COMPREHENSIVE DATASET OVERVIEW
Dataset Name............. Breast Cancer Wisconsin (Diagnostic)
Shape.................... 2500 rows × 11 columns
Memory Usage............. 0.36 MB
Total Missing Values..... 0
Missing Percentage....... 0.00%
Duplicate Rows........... 129
Numeric Columns.......... 10
Categorical Columns...... 1


In [10]:
# Code Chunk ID: CHUNK_2_1_5_B
# Enhanced Column Analysis - OUTPUT TO FILE
print("📊 DETAILED COLUMN ANALYSIS (SAVING TO FILE)")
print("=" * 50)

column_analysis = pd.DataFrame({
    'Column': data.columns,
    'Data_Type': data.dtypes.astype(str),
    'Unique_Values': [data[col].nunique() for col in data.columns],
    'Missing_Count': [data[col].isnull().sum() for col in data.columns],
    'Missing_Percent': [f"{(data[col].isnull().sum()/len(data)*100):.2f}%" for col in data.columns],
    'Min_Value': [data[col].min() if data[col].dtype in ['int64', 'float64'] else 'N/A' for col in data.columns],
    'Max_Value': [data[col].max() if data[col].dtype in ['int64', 'float64'] else 'N/A' for col in data.columns]
})

# Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
results_path = get_results_path(DATASET_IDENTIFIER, 2)
os.makedirs(results_path, exist_ok=True)
csv_file = f'{results_path}/column_analysis.csv'
column_analysis.to_csv(csv_file, index=False)

print(f"📊 Column analysis table saved to {csv_file}")
print(f"📊 Analysis completed for {len(data.columns)} features")

📊 DETAILED COLUMN ANALYSIS (SAVING TO FILE)
📊 Column analysis table saved to results/liver-train/2025-12-16/Section-2/column_analysis.csv
📊 Analysis completed for 11 features


This code conducts an enhanced analysis of the target variable within a dataset. It computes the counts and percentages of target classes, organizing the results into a DataFrame called `target_summary`, which distinguishes between benign and malignant classes if applicable. The class balance is assessed by calculating a balance ratio, with outputs indicating whether the dataset is balanced, moderately imbalanced, or highly imbalanced. If the specified target column is not found, it displays a warning and lists available columns in the dataset.

In [11]:
# Code Chunk ID: CHUNK_2_1_5_C
# Enhanced Target Variable Analysis - OUTPUT TO FILE
print("🎯 TARGET VARIABLE ANALYSIS (SAVING TO FILE)")
print("=" * 40)

if target_column in data.columns:
    target_counts = data[target_column].value_counts().sort_index()
    target_props = data[target_column].value_counts(normalize=True).sort_index() * 100
    
    target_summary = pd.DataFrame({
        'Class': target_counts.index,
        'Count': target_counts.values,
        'Percentage': [f"{prop:.1f}%" for prop in target_props.values],
        'Description': ['Benign (Non-cancerous)', 'Malignant (Cancerous)'] if len(target_counts) == 2 else [f'Class {i}' for i in target_counts.index]
    })
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    csv_file = f'{results_path}/target_analysis.csv'
    target_summary.to_csv(csv_file, index=False)
    
    # Calculate class balance metrics
    balance_ratio = target_counts.min() / target_counts.max()
    
    # Save balance metrics to separate file
    balance_metrics = pd.DataFrame({
        'Metric': ['Class_Balance_Ratio', 'Dataset_Balance_Category'],
        'Value': [f"{balance_ratio:.3f}", 
                 'Balanced' if balance_ratio > 0.8 else 'Moderately Imbalanced' if balance_ratio > 0.5 else 'Highly Imbalanced']
    })
    balance_file = f'{results_path}/target_balance_metrics.csv'
    balance_metrics.to_csv(balance_file, index=False)
    
    print(f"📊 Target variable analysis saved to {csv_file}")
    print(f"📊 Class balance metrics saved to {balance_file}")
    print(f"📊 Class Balance Ratio: {balance_ratio:.3f}")
    print(f"📊 Dataset Balance: {'Balanced' if balance_ratio > 0.8 else 'Moderately Imbalanced' if balance_ratio > 0.5 else 'Highly Imbalanced'}")
    
else:
    print(f"⚠️ Warning: Target column '{target_column}' not found!")
    print(f"Available columns: {list(data.columns)}")

🎯 TARGET VARIABLE ANALYSIS (SAVING TO FILE)
📊 Target variable analysis saved to results/liver-train/2025-12-16/Section-2/target_analysis.csv
📊 Class balance metrics saved to results/liver-train/2025-12-16/Section-2/target_balance_metrics.csv
📊 Class Balance Ratio: 0.397
📊 Dataset Balance: Highly Imbalanced


This code provides enhanced visualizations of feature distributions in a dataset. It retrieves numeric columns, excluding the target variable, and generates histograms for each numeric feature, displaying them in a grid layout. The histograms are enhanced with options for density, color, and grid lines to improve readability. If no numeric features are found, a warning message is displayed; otherwise, the generated plots give insights into the distributions of the numeric features in the dataset.

In [12]:
# Code Chunk ID: CHUNK_2_1_5_D
# Enhanced Feature Distribution Visualizations - OUTPUT TO FILE
print("📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)")
print("=" * 40)

# Turn off interactive mode to prevent figures from displaying in notebook
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
plt.ioff()  # Turn off interactive mode

# Get numeric columns excluding target
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
if target_column in numeric_cols:
    numeric_cols.remove(target_column)

if numeric_cols:
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    # Use dataset name fallback for title
    dataset_name = DATASET_IDENTIFIER.title() if DATASET_IDENTIFIER else "Dataset"
    fig.suptitle(f'{dataset_name} - Feature Distributions', fontsize=16, fontweight='bold')
    
    # Handle different subplot configurations
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(numeric_cols):
        if i < len(axes):
            # Enhanced histogram
            axes[i].hist(data[col], bins=30, alpha=0.7, color='skyblue', 
                        edgecolor='black', density=True)
            
            axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Density')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for j in range(len(numeric_cols), len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    plot_file = f'{results_path}/feature_distributions.png'
    plt.savefig(plot_file, dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory
    
    print(f"📊 Feature distribution plots saved to {plot_file}")
    print(f"📊 Distribution analysis completed for {len(numeric_cols)} numeric features")
else:
    print("⚠️ No numeric features found for visualization")

📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)
📊 Feature distribution plots saved to results/liver-train/2025-12-16/Section-2/feature_distributions.png
📊 Distribution analysis completed for 9 numeric features


This code conducts an enhanced correlation analysis of features within a dataset. It calculates the correlation matrix for numeric columns and includes the target variable if it is numeric, displaying the results in a heatmap for better visualization. The analysis identifies correlations with the target variable, categorizing each feature based on its correlation strength (strong, moderate, or weak) and presenting the findings in a DataFrame. If there are insufficient numeric features, a warning message is displayed, indicating that correlation analysis cannot be performed.

In [13]:
# Code Chunk ID: CHUNK_2_1_5_E
# Enhanced Correlation Analysis - OUTPUT TO FILE
print("🔍 CORRELATION ANALYSIS (SAVING TO FILE)")
print("=" * 30)

# Turn off interactive mode to prevent figures from displaying in notebook
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
plt.ioff()  # Turn off interactive mode

if len(numeric_cols) > 1:
    # Include target in correlation if numeric
    cols_for_corr = numeric_cols.copy()
    if data[target_column].dtype in ['int64', 'float64']:
        cols_for_corr.append(target_column)
    
    correlation_matrix = data[cols_for_corr].corr()
    
    # Enhanced correlation heatmap
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(correlation_matrix, 
                annot=True, 
                cmap='RdBu_r',
                center=0, 
                square=True, 
                linewidths=0.5,
                fmt='.3f',
                ax=ax)
    
    # Use dataset name fallback for title
    dataset_name = DATASET_IDENTIFIER.title() if DATASET_IDENTIFIER else "Dataset"
    ax.set_title(f'{dataset_name} - Feature Correlation Matrix', 
              fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    
    # Use new folder structure: results/dataset_identifier/YYYY-MM-DD/Section-2
    results_path = get_results_path(DATASET_IDENTIFIER, 2)
    os.makedirs(results_path, exist_ok=True)
    heatmap_file = f'{results_path}/correlation_heatmap.png'
    plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
    plt.close()  # Close the figure to free memory
    
    # Save correlation matrix to CSV
    corr_matrix_file = f'{results_path}/correlation_matrix.csv'
    correlation_matrix.to_csv(corr_matrix_file)
    
    print(f"🔍 Correlation heatmap saved to {heatmap_file}")
    print(f"🔍 Correlation matrix saved to {corr_matrix_file}")
    
    # Correlation with target analysis
    if target_column in correlation_matrix.columns:
        print("\n🔍 CORRELATIONS WITH TARGET VARIABLE (SAVING TO FILE)")
        print("=" * 45)
        
        target_corrs = correlation_matrix[target_column].abs().sort_values(ascending=False)
        target_corrs = target_corrs[target_corrs.index != target_column]
        
        corr_analysis = pd.DataFrame({
            'Feature': target_corrs.index,
            'Absolute_Correlation': target_corrs.values,
            'Raw_Correlation': [correlation_matrix.loc[feat, target_column] for feat in target_corrs.index],
            'Strength': ['Strong' if abs(corr) > 0.7 else 'Moderate' if abs(corr) > 0.3 else 'Weak' 
                        for corr in target_corrs.values]
        })
        
        # Save correlation analysis to CSV instead of displaying
        corr_analysis_file = f'{results_path}/target_correlations.csv'
        corr_analysis.to_csv(corr_analysis_file, index=False)
        
        print(f"🔍 Target correlation analysis saved to {corr_analysis_file}")
        print(f"📊 Correlation analysis completed for {len(target_corrs)} features")
    
else:
    print("⚠️ Insufficient numeric features for correlation analysis")

🔍 CORRELATION ANALYSIS (SAVING TO FILE)
🔍 Correlation heatmap saved to results/liver-train/2025-12-16/Section-2/correlation_heatmap.png
🔍 Correlation matrix saved to results/liver-train/2025-12-16/Section-2/correlation_matrix.csv

🔍 CORRELATIONS WITH TARGET VARIABLE (SAVING TO FILE)
🔍 Target correlation analysis saved to results/liver-train/2025-12-16/Section-2/target_correlations.csv
📊 Correlation analysis completed for 9 features


This code sets up global configuration variables for consistent evaluation across model evaluations. It checks for the existence of required variables, such as `data` and `target_column`, and raises an error if they are not defined. The code establishes global constants for the target column, results directory, and a copy of the original data while defining categorical columns, excluding the target. It then creates the results directory if it does not already exist and verifies that all necessary global variables are present, providing feedback on the setup's success.

In [14]:
# Code Chunk ID: CHUNK_2_1_5_F
# ============================================================================
# GLOBAL CONFIGURATION VARIABLES
# ============================================================================
# These variables are used across all sections for consistent evaluation

# Verify required variables exist before setting globals
if 'data' not in globals() or 'target_column' not in globals():
    raise ValueError("❌ ERROR: 'data' and 'target_column' must be defined before setting global variables. Please run the data loading cell first.")

# Set up global variables for use in all model evaluations
TARGET_COLUMN = target_column  # Use the target column from data loading

# 🔧 UPDATED: Preserve dataset-specific RESULTS_DIR that was set in CHUNK_005
# Don't override it with a generic path - maintain the structured approach
if 'RESULTS_DIR' not in globals() or RESULTS_DIR is None:
    # Fallback: reconstruct proper results directory structure
    RESULTS_DIR = f"results/{setup.DATASET_IDENTIFIER}/"
    print(f"⚠️  RESULTS_DIR was missing - using fallback: {RESULTS_DIR}")
else:
    print(f"✅ Using existing RESULTS_DIR: {RESULTS_DIR}")

data = data.copy()    # Create a copy of original data for evaluation functions

# Define categorical columns for all models
categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
if TARGET_COLUMN in categorical_columns:
    categorical_columns.remove(TARGET_COLUMN)  # Remove target from categorical list

# Apply user-specified categorical columns if provided
if 'CATEGORICAL_COLUMNS' in globals() and CATEGORICAL_COLUMNS:
    categorical_columns = CATEGORICAL_COLUMNS
    print(f"   • Using user-specified categorical columns: {categorical_columns}")
else:
    print(f"   • Auto-detected categorical columns: {categorical_columns}")

print("🔧 Global Configuration Summary:")
print(f"   • TARGET_COLUMN: {TARGET_COLUMN}")
print(f"   • RESULTS_DIR: {RESULTS_DIR}")
print(f"   • data shape: {data.shape}")
print(f"   • categorical_columns: {categorical_columns}")

# Create base results directory if it doesn't exist
import os
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"   • Created base results directory: {RESULTS_DIR}")
else:
    print(f"   • Base results directory already exists: {RESULTS_DIR}")

# Validate that all required variables are now available
required_vars = ['TARGET_COLUMN', 'RESULTS_DIR', 'data', 'categorical_columns']
missing_vars = [var for var in required_vars if var not in globals()]

if missing_vars:
    raise ValueError(f"❌ ERROR: Missing required global variables: {missing_vars}")
else:
    print("✅ All required global variables are now available for Section 3 evaluations")

✅ Using existing RESULTS_DIR: results/liver-train/
   • Using user-specified categorical columns: ['Gender of the patient']
🔧 Global Configuration Summary:
   • TARGET_COLUMN: Result
   • RESULTS_DIR: results/liver-train/
   • data shape: (2500, 11)
   • categorical_columns: ['Gender of the patient']
   • Base results directory already exists: results/liver-train/
✅ All required global variables are now available for Section 3 evaluations


In [15]:
data.head

<bound method NDFrame.head of        Age of the patient Gender of the patient  Total Bilirubin  \
13557                65.0                  Male              0.6   
6870                 22.0                Female              0.8   
27771                42.0                  Male              1.9   
2960                 38.0                  Male              1.8   
4771                  4.0                  Male              0.8   
...                   ...                   ...              ...   
14829                34.0                  Male              1.6   
21570                60.0                Female              3.3   
14514                32.0                  Male              7.3   
15104                73.0                  Male              2.8   
9845                 50.0                  Male              0.9   

       Direct Bilirubin  Alkphos Alkaline Phosphotase  \
13557               0.1                         176.0   
6870                0.2                

In [16]:
# ============================================================================
# SECTION 2 FINALIZATION: COMPLETE DATA PREPROCESSING PIPELINE
# ============================================================================
# Ensure all data preprocessing is complete and save final processed dataset

print("🎯 SECTION 2 FINALIZATION: COMPLETE DATA PREPROCESSING")
print("=" * 80)

# ============================================================================
# STEP 1: Apply smart categorical preprocessing
# ============================================================================
print("\n📊 STEP 1: Smart Categorical Data Preprocessing")
print("-" * 50)

# Apply our enhanced categorical preprocessing if not already applied
try:
    data_processed, categorical_cols_processed, encoders_processed = clean_and_preprocess_data(
        data, categorical_columns
    )
    
    print(f"✅ Smart categorical preprocessing completed:")
    print(f"   • Processed shape: {data_processed.shape}")
    print(f"   • Categorical columns: {categorical_cols_processed}")
    print(f"   • Missing values: {data_processed.isnull().sum().sum()}")
    
    # Update our data with processed version
    data = data_processed
    categorical_columns = categorical_cols_processed
    
except Exception as e:
    print(f"⚠️ Categorical preprocessing warning: {e}")
    print("   Using existing data as-is")

# ============================================================================
# STEP 2: Final data validation and summary
# ============================================================================
print("\n📊 STEP 2: Final Data Validation")
print("-" * 50)

# Display comprehensive categorical summary
display_categorical_summary(data, categorical_columns, TARGET_COLUMN)

# Final data quality checks
print(f"\n🔍 Final Data Quality Report:")
print(f"   • Shape: {data.shape}")
print(f"   • Missing values: {data.isnull().sum().sum()}")
print(f"   • Target column: {TARGET_COLUMN}")
print(f"   • Target distribution: {data[TARGET_COLUMN].value_counts().to_dict()}")
print(f"   • Categorical columns: {categorical_columns}")
print(f"   • Numeric columns: {len(data.select_dtypes(include=[np.number]).columns)}")

# ============================================================================
# STEP 3: Save processed dataset for Sections 3 & 4
# ============================================================================
print(f"\n💾 STEP 3: Saving Processed Dataset")
print("-" * 50)

# Ensure directory exists
import os
os.makedirs("data", exist_ok=True)

# Save the final processed dataset
processed_dataset_path = "data/liver_train_final_processed.csv"
data.to_csv(processed_dataset_path, index=False)

print(f"✅ Final processed dataset saved to: {processed_dataset_path}")
print(f"   • This dataset will be used in Sections 3 and 4")
print(f"   • All preprocessing, imputation, and categorical encoding applied")
print(f"   • Ready for synthetic data generation and evaluation")

# ============================================================================
# STEP 4: Set globals for Sections 3 & 4
# ============================================================================
print(f"\n🔧 STEP 4: Global Variables for Sections 3 & 4")
print("-" * 50)

# Set the path for Sections 3 & 4 to use
PROCESSED_DATASET_PATH = processed_dataset_path

print(f"✅ Global variables ready:")
print(f"   • TARGET_COLUMN: {TARGET_COLUMN}")
print(f"   • RESULTS_DIR: {RESULTS_DIR}")
print(f"   • PROCESSED_DATASET_PATH: {PROCESSED_DATASET_PATH}")
print(f"   • categorical_columns: {categorical_columns}")

print("\n" + "=" * 80)
print("🚀 SECTION 2 COMPLETE - Data Ready for Synthetic Generation!")
print("🎯 Sections 3 & 4 will use the processed dataset for consistent results")
print("=" * 80)

🎯 SECTION 2 FINALIZATION: COMPLETE DATA PREPROCESSING

📊 STEP 1: Smart Categorical Data Preprocessing
--------------------------------------------------
[DATA_CLEANING] Processing 2500 rows, 11 columns
[DATA_CLEANING] Categorical columns: ['Gender of the patient']
[DATA_CLEANING] Binary encoded column 'Gender of the patient' (2 values: ['Male', 'Female'] → 0/1)
[DATA_CLEANING] Data cleaning completed successfully
[DATA_CLEANING] Final shape: (2500, 11)
[DATA_CLEANING] Data types: {'Age of the patient': dtype('float64'), 'Gender of the patient': dtype('int32'), 'Total Bilirubin': dtype('float64'), 'Direct Bilirubin': dtype('float64'), 'Alkphos Alkaline Phosphotase': dtype('float64'), 'Sgpt Alamine Aminotransferase': dtype('float64'), 'Sgot Aspartate Aminotransferase': dtype('float64'), 'Total Protiens': dtype('float64'), 'ALB Albumin': dtype('float64'), 'A/G Ratio Albumin and Globulin Ratio': dtype('float64'), 'Result': dtype('int64')}
✅ Smart categorical preprocessing completed:
   • P

In [17]:
data.head

<bound method NDFrame.head of        Age of the patient  Gender of the patient  Total Bilirubin  \
13557                65.0                      1              0.6   
6870                 22.0                      0              0.8   
27771                42.0                      1              1.9   
2960                 38.0                      1              1.8   
4771                  4.0                      1              0.8   
...                   ...                    ...              ...   
14829                34.0                      1              1.6   
21570                60.0                      0              3.3   
14514                32.0                      1              7.3   
15104                73.0                      1              2.8   
9845                 50.0                      1              0.9   

       Direct Bilirubin  Alkphos Alkaline Phosphotase  \
13557               0.1                         176.0   
6870                0.2    

In [18]:
column_names = data.columns.tolist()

print(column_names)

['Age of the patient', 'Gender of the patient', 'Total Bilirubin', 'Direct Bilirubin', 'Alkphos Alkaline Phosphotase', 'Sgpt Alamine Aminotransferase', 'Sgot Aspartate Aminotransferase', 'Total Protiens', 'ALB Albumin', 'A/G Ratio Albumin and Globulin Ratio', 'Result']


In [19]:
# Convert to log scale
data['Total Bilirubin'] = np.log1p(data['Total Bilirubin'])
data['Direct Bilirubin'] = np.log1p(data['Direct Bilirubin'])
data['Alkphos Alkaline Phosphotase'] = np.log1p(data['Alkphos Alkaline Phosphotase'])
data['Sgpt Alamine Aminotransferase'] = np.log1p(data['Sgpt Alamine Aminotransferase'])
data['Sgot Aspartate Aminotransferase'] = np.log1p(data['Sgot Aspartate Aminotransferase'])


In [20]:
# Code Chunk ID: CHUNK_2_1_5_G
# Enhanced Feature Distribution Visualizations - OUTPUT TO FILE
print("📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)")
print("=" * 60)

# Create results directory for this section
# results_path = setup.get_results_path("Section-2")
# os.makedirs(results_path, exist_ok=True)

# 📊 DISTRIBUTION VISUALIZATION WITH FILE OUTPUT
def create_enhanced_distribution_plots_to_file(df, target_col, results_path):
    """Create comprehensive distribution plots and save to files"""
    
    # Separate numeric and categorical columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in numeric_cols:
        numeric_cols.remove(target_col)
    
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)
    
    print(f"📈 Analyzing {len(numeric_cols)} numeric and {len(categorical_cols)} categorical features")
    
    # 1. NUMERIC FEATURE DISTRIBUTIONS
    if numeric_cols:
        n_numeric = len(numeric_cols)
        n_cols = min(3, n_numeric)
        n_rows = (n_numeric + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, col in enumerate(numeric_cols):
            if i < len(axes):
                ax = axes[i]
                
                # Distribution plot with KDE
                data[col].hist(bins=30, alpha=0.7, ax=ax, density=True)
                data[col].plot(kind='kde', ax=ax, secondary_y=False)
                
                ax.set_title(f'{col} Distribution')
                ax.set_xlabel(col)
                ax.set_ylabel('Density')
                
                # Add statistics
                mean_val = data[col].mean()
                std_val = data[col].std()
                ax.axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_val:.2f}')
                ax.legend()
        
        # Hide unused subplots
        for i in range(n_numeric, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        numeric_dist_path = os.path.join(results_path, 'numeric_distributions.png')
        plt.savefig(numeric_dist_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"💾 Numeric distributions saved: {numeric_dist_path}")
    
    # 2. CATEGORICAL FEATURE DISTRIBUTIONS
    if categorical_cols:
        n_categorical = len(categorical_cols)
        n_cols = min(2, n_categorical)
        n_rows = (n_categorical + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, col in enumerate(categorical_cols):
            if i < len(axes):
                ax = axes[i]
                
                value_counts = data[col].value_counts()
                value_counts.plot(kind='bar', ax=ax)
                
                ax.set_title(f'{col} Distribution')
                ax.set_xlabel(col)
                ax.set_ylabel('Count')
                ax.tick_params(axis='x', rotation=45)
                
                # Add value labels on bars
                for j, v in enumerate(value_counts.values):
                    ax.text(j, v + 0.1, str(v), ha='center', va='bottom')
        
        # Hide unused subplots
        for i in range(n_categorical, len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        categorical_dist_path = os.path.join(results_path, 'categorical_distributions.png')
        plt.savefig(categorical_dist_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"💾 Categorical distributions saved: {categorical_dist_path}")

# Execute the enhanced visualization
create_enhanced_distribution_plots_to_file(data, TARGET_COLUMN, results_path)

print(f"✅ Enhanced distribution analysis complete - files saved to: {results_path}")

📊 FEATURE DISTRIBUTION ANALYSIS (SAVING TO FILE)
📈 Analyzing 10 numeric and 0 categorical features
💾 Numeric distributions saved: results/liver-train/2025-12-16/Section-2\numeric_distributions.png
✅ Enhanced distribution analysis complete - files saved to: results/liver-train/2025-12-16/Section-2


## 3 Demo All Models with Default Parameters

### 3.1 Demos

Each model is run with default parameters for purposes of testing.

#### 3.1.1 CTGAN Demo

In [21]:
# Code Chunk ID: CHUNK_3_1_1_A
import time
try:
    print("🔄 CTGAN Demo - Default Parameters")
    print("=" * 500)
    
    # Import and initialize CTGAN model using ModelFactory
    from src.models.model_factory import ModelFactory
    
    ctgan_model = ModelFactory.create("ctgan", random_state=42)
    
    # Define demo parameters for quick execution
    demo_params = {
        'epochs': 500,
        'batch_size': 100,
        'generator_dim': (128, 128),
        'discriminator_dim': (128, 128)
    }
    
    # Train with demo parameters
    print("Training CTGAN with demo parameters...")
    start_time = time.time()
    
    # Auto-detect discrete columns
    discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
    
    ctgan_model.train(data, discrete_columns=discrete_columns, **demo_params)
    train_time = time.time() - start_time
    
    # Generate synthetic data
    demo_samples = len(data)  # Same size as original dataset
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_ctgan = ctgan_model.generate(demo_samples)
    
    print(f"✅ CTGAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctgan)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_ctgan.shape}")
    
    # Store for later use in comprehensive evaluation
    demo_results_ctgan = {
        'model': ctgan_model,
        'synthetic_data': synthetic_data_ctgan,
        'training_time': train_time,
        'parameters_used': demo_params
    }
    
except ImportError as e:
    print(f"❌ CTGAN not available: {e}")
    print(f"   Please ensure CTGAN dependencies are installed")
except Exception as e:
    print(f"❌ Error during CTGAN demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTGAN Demo - Default Parameters
Training CTGAN with demo parameters...


Gen. (-2.53) | Discrim. (0.18): 100%|██████████| 500/500 [01:49<00:00,  4.56it/s] 

Generating 2500 synthetic samples...
✅ CTGAN Demo completed successfully!
   - Training time: 123.27 seconds
   - Generated samples: 2500
   - Original data shape: (2500, 11)
   - Synthetic data shape: (2500, 11)


#### 3.1.2 CTAB-GAN Demo

In [22]:
# Code Chunk ID: CHUNK_3_1_2_A
try:
    print("🔄 CTAB-GAN Demo - Default Parameters")
    print("=" * 50)
    
    # Check CTABGAN availability (imported from setup.py)
    if not CTABGAN_AVAILABLE:
        raise ImportError("CTAB-GAN not available - clone and install CTAB-GAN repository")
    
    # Initialize CTAB-GAN model (already defined in notebook)
    ctabgan_model = CTABGANModel()
    print("✅ CTAB-GAN model initialized successfully")
    
    # Record start time
    start_time = time.time()
    
    # Train the model with demo parameters
    print("🚀 Training CTAB-GAN model (epochs=500)...")
    ctabgan_model.fit(data, categorical_columns=None, target_column=target_column)
    
    # Record training time
    train_time = time.time() - start_time
    
    # Generate synthetic data
    print("🎯 Generating synthetic data...")
    synthetic_data_ctabgan = ctabgan_model.generate(len(data))
    
    # Display results
    print("✅ CTAB-GAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctabgan)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ctabgan.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ctabgan, 'head'):
        # It's a DataFrame
        print(synthetic_data_ctabgan.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ctabgan[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ CTAB-GAN not available: {e}")
    print(f"   Please ensure CTAB-GAN dependencies are installed")
    print(f"   Note: CTABGAN_AVAILABLE = {globals().get('CTABGAN_AVAILABLE', 'undefined')}")
except Exception as e:
    print(f"❌ Error during CTAB-GAN demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTAB-GAN Demo - Default Parameters
✅ CTAB-GAN model initialized successfully
🚀 Training CTAB-GAN model (epochs=500)...
[CTABGAN] Applying comprehensive data preprocessing...
[DATA_CLEANING] Processing 2500 rows, 11 columns
[DATA_CLEANING] Categorical columns: []
[DATA_CLEANING] Data cleaning completed successfully
[DATA_CLEANING] Final shape: (2500, 11)
[DATA_CLEANING] Data types: {'Age of the patient': dtype('float64'), 'Gender of the patient': dtype('int32'), 'Total Bilirubin': dtype('float64'), 'Direct Bilirubin': dtype('float64'), 'Alkphos Alkaline Phosphotase': dtype('float64'), 'Sgpt Alamine Aminotransferase': dtype('float64'), 'Sgot Aspartate Aminotransferase': dtype('float64'), 'Total Protiens': dtype('float64'), 'ALB Albumin': dtype('float64'), 'A/G Ratio Albumin and Globulin Ratio': dtype('float64'), 'Result': dtype('int64')}
[CTABGAN] Using categorical columns: []
[CTABGAN] Data shape after preprocessing: (2500, 11)
[CTABGAN] Training CTAB-GAN for 100 epochs...


100%|██████████| 100/100 [05:02<00:00,  3.02s/it]


[OK] CTAB-GAN training completed successfully
🎯 Generating synthetic data...
[CTABGAN] Generated 2500 raw synthetic samples
[OK] CTAB-GAN generation completed: (2500, 11)
✅ CTAB-GAN Demo completed successfully!
   - Training time: 305.29 seconds
   - Generated samples: 2500
   - Original shape: (2500, 11)
   - Synthetic shape: (2500, 11)

📊 Sample of generated data:
   Age of the patient  Gender of the patient  Total Bilirubin  \
0           67.053273               0.994253         0.879644   
1           51.944053              -0.072101         0.377022   
2           85.487801              -0.100302         0.321048   
3           58.311851               0.968518         0.310597   
4           73.033612               0.979851         2.316110   

   Direct Bilirubin  Alkphos Alkaline Phosphotase  \
0          0.514668                      5.355018   
1          0.515342                      5.956012   
2          0.219543                      5.555950   
3          0.232064         

#### 3.1.3 CTAB-GAN+ Demo

In [23]:
# Code Chunk ID: CHUNK_3_1_3_A
try:
    print("🔄 CTAB-GAN+ Demo - Default Parameters")
    print("=" * 50)
    
    # Check CTABGAN+ availability with fallback
    try:
        ctabganplus_available = CTABGANPLUS_AVAILABLE
    except NameError:
        print("⚠️  CTABGANPLUS_AVAILABLE variable not defined - checking direct import...")
        try:
            # Try to check if CTABGANPLUS (the imported class) exists
            from model.ctabgan import CTABGAN as CTABGANPLUS
            ctabganplus_available = True
            print("✅ CTAB-GAN+ import check successful")
        except ImportError:
            ctabganplus_available = False
            print("❌ CTAB-GAN+ import check failed")
    
    if not ctabganplus_available:
        raise ImportError("CTAB-GAN+ not available - clone and install CTAB-GAN+ repository")
    
    # Initialize CTAB-GAN+ model with epochs parameter in constructor
    ctabganplus_model = CTABGANPlusModel(epochs=500)
    print("✅ CTAB-GAN+ model initialized successfully")
    
    # Record start time
    start_time = time.time()
    
    # Train the model (epochs already set in constructor)
    print("🚀 Training CTAB-GAN+ model (epochs=500)...")
    ctabganplus_model.fit(data)
    
    # Record training time
    train_time = time.time() - start_time
    
    # Generate synthetic data
    print("🎯 Generating synthetic data...")
    synthetic_data_ctabganplus = ctabganplus_model.generate(len(data))
    
    # Display results
    print("✅ CTAB-GAN+ Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ctabganplus)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ctabganplus.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ctabganplus, 'head'):
        # It's a DataFrame
        print(synthetic_data_ctabganplus.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ctabganplus[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ CTAB-GAN+ not available: {e}")
    print(f"   Please ensure CTAB-GAN+ dependencies are installed")
except Exception as e:
    print(f"❌ Error during CTAB-GAN+ demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CTAB-GAN+ Demo - Default Parameters
✅ CTAB-GAN+ model initialized successfully
🚀 Training CTAB-GAN+ model (epochs=500)...
[CTABGAN+] Applying comprehensive data preprocessing...
[DATA_CLEANING] Processing 2500 rows, 11 columns
[DATA_CLEANING] Categorical columns: []
[DATA_CLEANING] Data cleaning completed successfully
[DATA_CLEANING] Final shape: (2500, 11)
[DATA_CLEANING] Data types: {'Age of the patient': dtype('float64'), 'Gender of the patient': dtype('int32'), 'Total Bilirubin': dtype('float64'), 'Direct Bilirubin': dtype('float64'), 'Alkphos Alkaline Phosphotase': dtype('float64'), 'Sgpt Alamine Aminotransferase': dtype('float64'), 'Sgot Aspartate Aminotransferase': dtype('float64'), 'Total Protiens': dtype('float64'), 'ALB Albumin': dtype('float64'), 'A/G Ratio Albumin and Globulin Ratio': dtype('float64'), 'Result': dtype('int64')}
[CTABGAN+] Using categorical columns: []
[CTABGAN+] Data shape after preprocessing: (2500, 11)
[CTABGAN+] Using Classification with target: Result

100%|██████████| 1/1 [00:01<00:00,  1.95s/it]

Finished training in 4.562059164047241  seconds.
[OK] CTAB-GAN+ training completed successfully
🎯 Generating synthetic data...
[CTABGAN+] Generated 2500 raw synthetic samples
[OK] CTAB-GAN+ generation completed: (2500, 11)
✅ CTAB-GAN+ Demo completed successfully!
   - Training time: 4.67 seconds
   - Generated samples: 2500
   - Original shape: (2500, 11)
   - Synthetic shape: (2500, 11)

📊 Sample of generated data:
   Age of the patient  Gender of the patient  Total Bilirubin  \
0           56.990499                      1         1.286060   
1           21.406396                      0         0.988874   
2           44.114743                      1         0.760344   
3           21.731748                      1         2.253736   
4           44.451687                      0         1.289172   

   Direct Bilirubin  Alkphos Alkaline Phosphotase  \
0          0.938931                      4.691284   
1          0.980738                      5.612023   
2          0.336650           

#### 3.1.4 GANerAid Demo

In [24]:
# Code Chunk ID: CHUNK_3_1_4_A
try:
    print("🔄 GANerAid Demo - Default Parameters")
    print("=" * 50)
    
    # Check GANerAid availability with fallback
    try:
        ganeraid_available = GANERAID_AVAILABLE
        GANerAidModel  # Test if the class is available
    except NameError:
        print("⚠️ GANerAidModel not available - checking import...")
        try:
            # Try to import GANerAidModel
            from src.models.implementations.ganeraid_model import GANerAidModel
            ganeraid_available = True
            print("✅ GANerAidModel import successful")
        except ImportError:
            ganeraid_available = False
            print("❌ GANerAidModel import failed")
    
    if not ganeraid_available:
        raise ImportError("GANerAid not available - please install GANerAid dependencies")
    
    # Initialize GANerAid model
    ganeraid_model = GANerAidModel()
    print("✅ GANerAid model initialized successfully")
    
    # Define demo_samples variable for synthetic data generation
    demo_samples = len(data)  # Same size as original dataset
    
    # Train with minimal parameters for demo
    demo_params = {'epochs': 500, 'batch_size': 100}
    start_time = time.time()
    ganeraid_model.train(data, **demo_params)  # GANerAid uses train method
    train_time = time.time() - start_time
    
    # Generate synthetic data
    synthetic_data_ganeraid = ganeraid_model.generate(demo_samples)
    
    print(f"✅ GANerAid Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_ganeraid)}")
    print(f"   - Original shape: {data.shape}")
    print(f"   - Synthetic shape: {synthetic_data_ganeraid.shape}")
    
    # Show sample of synthetic data with proper handling for both DataFrame and array
    print(f"\n📊 Sample of generated data:")
    if hasattr(synthetic_data_ganeraid, 'head'):
        # It's a DataFrame
        print(synthetic_data_ganeraid.head())
    else:
        # It's likely a numpy array
        print("First 5 rows of synthetic data:")
        print(synthetic_data_ganeraid[:5])
    print("=" * 50)
    
except ImportError as e:
    print(f"❌ GANerAid not available: {e}")
    print(f"   Please ensure GANerAid dependencies are installed")
except Exception as e:
    print(f"❌ Error during GANerAid demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 GANerAid Demo - Default Parameters
✅ GANerAid model initialized successfully
Initialized gan with the following parameters: 
lr_d = 0.0005
lr_g = 0.0005
hidden_feature_space = 200
batch_size = 100
nr_of_rows = 25
binary_noise = 0.2
Start training of gan for 500 epochs


100%|██████████| 500/500 [01:31<00:00,  5.47it/s, loss=d error: 0.2440449222922325 --- g error 5.419250011444092]   


Generating 2500 samples
✅ GANerAid Demo completed successfully!
   - Training time: 91.92 seconds
   - Generated samples: 2500
   - Original shape: (2500, 11)
   - Synthetic shape: (2500, 11)

📊 Sample of generated data:
   Age of the patient  Gender of the patient  Total Bilirubin  \
0           43.224773                      1         0.561600   
1           43.300301                      1         0.690705   
2           40.214462                      1         1.878308   
3           35.570881                      1         0.511002   
4           45.709743                      1         0.612893   

   Direct Bilirubin  Alkphos Alkaline Phosphotase  \
0          0.241346                      5.155302   
1          0.268045                      5.184514   
2          1.265575                      5.240814   
3          0.185099                      4.790081   
4          0.224148                      5.048146   

   Sgpt Alamine Aminotransferase  Sgot Aspartate Aminotransferase  \


#### 3.1.5 CopulaGAN Demo

In [25]:
# Code Chunk ID: CHUNK_3_1_5_A
try:
    print("🔄 CopulaGAN Demo - Default Parameters")
    print("=" * 50)
    
    # Import and initialize CopulaGAN model using ModelFactory
    from src.models.model_factory import ModelFactory
    
    copulagan_model = ModelFactory.create("copulagan", random_state=42)
    
    # Define demo parameters optimized for CopulaGAN
    demo_params = {
        'epochs': 500,
        'batch_size': 100,
        'generator_dim': (128, 128),
        'discriminator_dim': (128, 128),
        'default_distribution': 'beta',  # Good for bounded data
        'enforce_min_max_values': True
    }
    
    # Train with demo parameters
    print("Training CopulaGAN with demo parameters...")
    start_time = time.time()
    
    # Auto-detect discrete columns for CopulaGAN
    discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
    
    copulagan_model.train(data, discrete_columns=discrete_columns, **demo_params)
    train_time = time.time() - start_time
    
    # Generate synthetic data
    demo_samples = len(data)  # Same size as original dataset
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_copulagan = copulagan_model.generate(demo_samples)
    
    print(f"✅ CopulaGAN Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_copulagan)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_copulagan.shape}")
    print(f"   - Distribution used: {demo_params['default_distribution']}")
    
    # Store for later use in comprehensive evaluation
    demo_results_copulagan = {
        'model': copulagan_model,
        'synthetic_data': synthetic_data_copulagan,
        'training_time': train_time,
        'parameters_used': demo_params
    }
    
except ImportError as e:
    print(f"❌ CopulaGAN not available: {e}")
    print(f"   Please ensure CopulaGAN dependencies are installed")
except Exception as e:
    print(f"❌ Error during CopulaGAN demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 CopulaGAN Demo - Default Parameters
Training CopulaGAN with demo parameters...
Generating 2500 synthetic samples...
✅ CopulaGAN Demo completed successfully!
   - Training time: 362.52 seconds
   - Generated samples: 2500
   - Original data shape: (2500, 11)
   - Synthetic data shape: (2500, 11)
   - Distribution used: beta


#### 3.1.6 TVAE Demo

In [26]:
# Code Chunk ID: CHUNK_3_1_6_A
try:
    print("🔄 TVAE Demo - Default Parameters")
    print("=" * 50)
    
    # Import and initialize TVAE model using ModelFactory
    from src.models.model_factory import ModelFactory
    
    tvae_model = ModelFactory.create("tvae", random_state=42)
    
    # Define demo parameters optimized for TVAE
    demo_params = {
        'epochs': 50,
        'batch_size': 100,
        'compress_dims': (128, 128),
        'decompress_dims': (128, 128),
        'l2scale': 1e-5,
        'loss_factor': 2,
        'learning_rate': 1e-3  # VAE-specific learning rate
    }
    
    # Train with demo parameters
    print("Training TVAE with demo parameters...")
    start_time = time.time()
    
    # Auto-detect discrete columns for TVAE
    discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
    
    tvae_model.train(data, discrete_columns=discrete_columns, **demo_params)
    train_time = time.time() - start_time
    
    # Generate synthetic data
    demo_samples = len(data)  # Same size as original dataset
    print(f"Generating {demo_samples} synthetic samples...")
    synthetic_data_tvae = tvae_model.generate(demo_samples)
    
    print(f"✅ TVAE Demo completed successfully!")
    print(f"   - Training time: {train_time:.2f} seconds")
    print(f"   - Generated samples: {len(synthetic_data_tvae)}")
    print(f"   - Original data shape: {data.shape}")
    print(f"   - Synthetic data shape: {synthetic_data_tvae.shape}")
    print(f"   - VAE architecture: compress{demo_params['compress_dims']} → decompress{demo_params['decompress_dims']}")
    
    # Store for later use in comprehensive evaluation
    demo_results_tvae = {
        'model': tvae_model,
        'synthetic_data': synthetic_data_tvae,
        'training_time': train_time,
        'parameters_used': demo_params
    }
    
except ImportError as e:
    print(f"❌ TVAE not available: {e}")
    print(f"   Please ensure TVAE dependencies are installed")
except Exception as e:
    print(f"❌ Error during TVAE demo: {str(e)}")
    print("   Check model implementation and data compatibility")
    import traceback
    traceback.print_exc()

🔄 TVAE Demo - Default Parameters
Training TVAE with demo parameters...
Generating 2500 synthetic samples...
✅ TVAE Demo completed successfully!
   - Training time: 15.88 seconds
   - Generated samples: 2500
   - Original data shape: (2500, 11)
   - Synthetic data shape: (2500, 11)
   - VAE architecture: compress(128, 128) → decompress(128, 128)


### 3.2 Batch Process

This section outputs figures and graphics from models run in 3.1. The next chunk will output results for whatever subsections of 3.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 3.1.

In [27]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3 - BATCH EVALUATION FOR ALL TRAINED MODELS
# Standardized evaluation using enhanced batch evaluation system
# ============================================================================

print("🔍 SECTION 3 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

section3_results = evaluate_all_available_models(
    section_number=3,
    scope=globals(),  # Pass notebook scope to access synthetic data variables
    models_to_evaluate=None,  # Evaluate all available models
    real_data=None,  # Will use 'data' from scope
    target_col=None   # Will use 'target_column' from scope
)

if section3_results:
    print(f"\n🎉 SECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"📊 Evaluated {len(section3_results)} models successfully")
    print(f"📁 All results saved to organized folder structure")
    
    # Show quick summary of best performing models
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\n🏆 RANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\n⚠️ No models available for evaluation")
    print("   Train some models first in previous sections")

🔍 SECTION 3 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: liver-train
[INFO] Target column: Result
[INFO] Variable pattern: standard
[INFO] Found 6 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] GANerAid
   [OK] CopulaGAN
   [OK] TVAE

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results\liver-train\2025-12-16\Section-3\CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.869

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.013
   [CHART] Explained Variance (PC1, PC2): 0.316, 0.180

[3] DISTRIBUTION SIMILARITY
-----------------------

WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


[SUCCESS] ROC curves saved: results/liver-train/2025-12-16/Section-3\trts_roc_curves.png

GENERATING PRECISION-RECALL CURVES
[WARNING] Could not generate PR curve for CTGAN TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for CTABGANPLUS TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for GANerAid TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for CopulaGAN TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for TVAE TRTR: y_true takes value in {1,

## 4: Hyperparameter Tuning for Each Model

### 4.1 Hyperparameter Optimization

#### 4.1.1 CTGAN Hyperparameter Optimization

In [28]:
# ============================================================================
# SECTION 4 DATA LOADING: USE PROCESSED DATASET FROM SECTION 2
# ============================================================================
# Load the final processed dataset saved from Section 2

print("🔧 SECTION 4: LOADING PROCESSED DATASET")
print("=" * 60)

# Load the processed dataset that was saved at the end of Section 2
if 'PROCESSED_DATASET_PATH' in globals():
    processed_path = PROCESSED_DATASET_PATH
else:
    processed_path = "data/liver_train_final_processed.csv"

print(f"📂 Loading processed dataset from: {processed_path}")

try:
    # Load the fully processed dataset
    data = pd.read_csv(processed_path)
    
    print(f"✅ Processed dataset loaded successfully!")
    print(f"📊 Shape: {data.shape}")
    print(f"📊 Missing values: {data.isnull().sum().sum()}")
    print(f"📊 Target column '{TARGET_COLUMN}' distribution:")
    print(data[TARGET_COLUMN].value_counts())
    
    # Verify categorical columns are properly processed
    if categorical_columns:
        print(f"\n📊 Categorical columns verification:")
        for col in categorical_columns:
            if col in data.columns:
                unique_vals = data[col].unique()
                print(f"   • {col}: {len(unique_vals)} unique values")
                if len(unique_vals) <= 5:
                    print(f"     Values: {list(unique_vals)}")
    
    print(f"\n✅ Section 4 ready with processed dataset!")
    print(f"   • All categorical data properly encoded")
    print(f"   • No missing values")
    print(f"   • Ready for hyperparameter optimization")
    
except FileNotFoundError:
    print(f"❌ ERROR: Processed dataset not found at {processed_path}")
    print(f"   Please ensure Section 2 has been run completely")
    print(f"   Section 2 should save the processed dataset automatically")
    raise

except Exception as e:
    print(f"❌ ERROR loading processed dataset: {e}")
    raise

print("=" * 60)

🔧 SECTION 4: LOADING PROCESSED DATASET
📂 Loading processed dataset from: data/liver_train_final_processed.csv
✅ Processed dataset loaded successfully!
📊 Shape: (2500, 11)
📊 Missing values: 0
📊 Target column 'Result' distribution:
Result
1    1790
2     710
Name: count, dtype: int64

📊 Categorical columns verification:
   • Gender of the patient: 2 unique values
     Values: [1, 0]

✅ Section 4 ready with processed dataset!
   • All categorical data properly encoded
   • No missing values
   • Ready for hyperparameter optimization


In [29]:
# Code Chunk ID: CHUNK_4_1_1_A
# CTGAN Hyperparameter Optimization Execution
# Complete optimization study with search space definition and execution

import optuna
import time
from datetime import datetime
import json
import pandas as pd

# ============================================================================
# SECTION 4.1: CTGAN HYPERPARAMETER OPTIMIZATION 
# ============================================================================

print("🔧 SECTION 4.1: CTGAN HYPERPARAMETER OPTIMIZATION")
print("=" * 80)

def ctgan_search_space(trial):
    """Define CTGAN hyperparameter search space with corrected PAC validation."""
    # Select batch size first
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 500, 1000])
    
    # PAC must be <= batch_size and batch_size must be divisible by PAC
    max_pac = min(20, batch_size)
    pac = trial.suggest_int('pac', 1, max_pac)
    
    """CTGAN search space definition based on working CTGAN and Optuna best practices."""
    return {
        'epochs': trial.suggest_int('epochs', 100, 1000, step=50),
        'batch_size': batch_size,
        'generator_lr': trial.suggest_loguniform('generator_lr', 5e-6, 5e-3),
        'discriminator_lr': trial.suggest_loguniform('discriminator_lr', 5e-6, 5e-3),
        'generator_dim': trial.suggest_categorical('generator_dim', [
            (128, 128),
            (256, 256), 
            (512, 256),
            (256, 512),
            (512, 512),
            (128, 256, 128),
            (256, 128, 64),
            (256, 512, 256)
        ]),
        'discriminator_dim': trial.suggest_categorical('discriminator_dim', [
            (128, 128),
            (256, 256),
            (256, 512), 
            (512, 256),
            (128, 256, 128),
            (256, 512, 256)
        ]),
        'pac': pac,
        'discriminator_steps': trial.suggest_int('discriminator_steps', 1, 5),
        'generator_decay': trial.suggest_loguniform('generator_decay', 1e-8, 1e-4),
        'discriminator_decay': trial.suggest_loguniform('discriminator_decay', 1e-8, 1e-4),
        'log_frequency': trial.suggest_categorical('log_frequency', [True, False]),
        'verbose': trial.suggest_categorical('verbose', [True])
    }

def ctgan_objective(trial):
    """CTGAN objective function with corrected PAC validation and fixed imports."""
    try:
        # Get hyperparameters from trial
        params = ctgan_search_space(trial)
        
        # CORRECTED PAC VALIDATION: Fix incompatible combinations if needed
        batch_size = params['batch_size']
        original_pac = params['pac']
        
        # Find the largest compatible PAC value <= original_pac
        compatible_pac = original_pac
        while batch_size % compatible_pac != 0 and compatible_pac > 1:
            compatible_pac -= 1
        
        if compatible_pac != original_pac:
            print(f"⚠️  Adjusted PAC from {original_pac} to {compatible_pac} for batch_size {batch_size}")
        
        params['pac'] = compatible_pac
        print(f"✅ PAC validation: {batch_size} % {compatible_pac} = {batch_size % compatible_pac}")

        print(f"\n🔄 CTGAN Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}, pac={params['pac']}, lr={params['generator_lr']:.2e}")
        
        # Import model factory
        from src.models.model_factory import ModelFactory
        
        # Create CTGAN model
        ctgan_model = ModelFactory.create("ctgan", random_state=42)
        
        print(f"🎯 Using target column: '{TARGET_COLUMN}'")
        print("✅ Using CTGAN from ctgan package")
        
        # Auto-detect discrete columns
        discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
        
        # Train model with trial parameters
        start_time = time.time()
        ctgan_model.train(data, discrete_columns=discrete_columns, **params)
        train_time = time.time() - start_time
        
        print(f"⏱️ Training completed in {train_time:.1f} seconds")
        
        # Generate synthetic data for evaluation
        synthetic_data = ctgan_model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        return 0.0

# Create and run optimization study
print(f"🔄 Creating CTGAN optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

ctgan_study = optuna.create_study(direction='maximize')
ctgan_study.optimize(ctgan_objective, n_trials=100)

# Extract and display results
best_trial = ctgan_study.best_trial
print(f"\n✅ CTGAN Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ CTGAN optimization completed successfully!")

[I 2025-12-16 14:15:20,795] A new study created in memory with name: no-name-1ecd33a4-e4ae-4149-b056-c4d6f899896f


🔧 SECTION 4.1: CTGAN HYPERPARAMETER OPTIMIZATION
🔄 Creating CTGAN optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2

⚠️  Adjusted PAC from 5 to 4 for batch_size 64
✅ PAC validation: 64 % 4 = 0

🔄 CTGAN Trial 1: epochs=500, batch_size=64, pac=4, lr=2.70e-04
🎯 Using target column: 'Result'
✅ Using CTGAN from ctgan package


Gen. (-2.99) | Discrim. (-0.19): 100%|██████████| 500/500 [01:58<00:00,  4.21it/s]


⏱️ Training completed in 126.9 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5326


[I 2025-12-16 14:17:28,865] Trial 0 finished with value: 0.6013632554844184 and parameters: {'batch_size': 64, 'pac': 5, 'epochs': 500, 'generator_lr': 0.00027015216522354054, 'discriminator_lr': 0.0005838802753007171, 'generator_dim': (256, 128, 64), 'discriminator_dim': (256, 512), 'discriminator_steps': 3, 'generator_decay': 1.428954723753875e-06, 'discriminator_decay': 1.5422340875568263e-05, 'log_frequency': True, 'verbose': True}. Best is trial 0 with value: 0.6013632554844184.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.6866
[CHART] Combined Score: 0.6180 (Similarity: 0.5722, Accuracy: 0.6866)
🎯 Trial 99 Results:
   • Combined Score: 0.6180
   • Similarity: 0.5722
   • Accuracy: 0.6866
⚠️  Adjusted PAC from 15 to 10 for batch_size 1000
✅ PAC validation: 1000 % 10 = 0

🔄 CTGAN Trial 100: epochs=900, batch_size=1000, pac=10, lr=6.18e-04
🎯 Using target column: 'Result'
✅ Using CTGAN from ctgan package


Gen. (-0.98) | Discrim. (0.07): 100%|██████████| 900/900 [01:56<00:00,  7.73it/s] 


⏱️ Training completed in 118.4 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5863


[I 2025-12-16 16:57:07,462] Trial 99 finished with value: 0.6360702715943973 and parameters: {'batch_size': 1000, 'pac': 15, 'epochs': 900, 'generator_lr': 0.000617660462064916, 'discriminator_lr': 0.000753859947562788, 'generator_dim': (128, 128), 'discriminator_dim': (256, 512), 'discriminator_steps': 1, 'generator_decay': 1.3018923527358575e-06, 'discriminator_decay': 6.593412820927046e-07, 'log_frequency': False, 'verbose': True}. Best is trial 92 with value: 0.6656534641462934.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.7107
[CHART] Combined Score: 0.6361 (Similarity: 0.5863, Accuracy: 0.7107)
🎯 Trial 100 Results:
   • Combined Score: 0.6361
   • Similarity: 0.5863
   • Accuracy: 0.7107

✅ CTGAN Optimization completed!
🏆 Best score: 0.6657
🔧 Best parameters:
   • batch_size: 1000
   • pac: 17
   • epochs: 900
   • generator_lr: 0.0005118246452015182
   • discriminator_lr: 0.0007252902764816536
   • generator_dim: (256, 256)
   • discriminator_dim: (256, 512)
   • discriminator_steps: 1
   • generator_decay: 9.334067695307696e-07
   • discriminator_decay: 3.583508568042387e-07
   • log_frequency: False
   • verbose: True
✅ CTGAN optimization completed successfully!


#### 4.1.2 CTAB-GAN Hyperparameter Optimization

In [30]:
# Code Chunk ID: CHUNK_4_1_2_A
# Import required libraries for CTAB-GAN optimization
import optuna
import numpy as np
import pandas as pd
from src.models.model_factory import ModelFactory
from src.evaluation.trts_framework import TRTSEvaluator

# ============================================================================
# CRITICAL FIX: Ensure clean, imputed subset data is loaded for CTAB-GAN
# ============================================================================
print("🔄 Reloading clean subset data for CTAB-GAN optimization...")
data = pd.read_csv("data/liver_train_final_processed.csv")
print(f"✅ Clean data loaded: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"✅ Missing values: {data.isnull().sum().sum()}")

# Validate data quality
if data.isnull().sum().sum() > 0:
    raise ValueError("ERROR: CTAB-GAN data still contains missing values!")
else:
    print("✅ Data validation passed: 0 missing values confirmed")

# CORRECTED CTAB-GAN Search Space (3 supported parameters only)
def ctabgan_search_space(trial):
    """Realistic CTAB-GAN hyperparameter space - ONLY supported parameters"""
    return {
        'epochs': trial.suggest_int('epochs', 100, 500, step=50),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),  # Remove 500 - not stable
        'test_ratio': trial.suggest_float('test_ratio', 0.15, 0.25, step=0.05),
        # REMOVED: class_dim, random_dim, num_channels (not supported by constructor)
    }

def ctabgan_objective(trial):
    """FINAL CORRECTED CTAB-GAN objective function with SCORE EXTRACTION FIX"""
    try:
        # Get realistic hyperparameters from trial
        params = ctabgan_search_space(trial)
        
        print(f"\n🔄 CTAB-GAN Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}, test_ratio={params['test_ratio']:.3f}")
        
        # Initialize CTAB-GAN using ModelFactory
        model = ModelFactory.create("ctabgan", random_state=42)
        
        # Only pass supported parameters to train()
        result = model.train(data, 
                           epochs=params['epochs'],
                           batch_size=params['batch_size'],
                           test_ratio=params['test_ratio'])

        print(f"🏋️ Training CTAB-GAN with corrected parameters...")

        # Generate synthetic data for evaluation
        synthetic_data = model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0

# Create and run optimization study
print(f"\n🔧 SECTION 4.2: CTAB-GAN HYPERPARAMETER OPTIMIZATION")
print("=" * 80)
print(f"🔄 Creating CTAB-GAN optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

# Create study and optimize
ctabgan_study = optuna.create_study(direction='maximize')
ctabgan_study.optimize(ctabgan_objective, n_trials=5)

# Extract and display results
best_trial = ctabgan_study.best_trial
print(f"\n✅ CTAB-GAN Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ CTAB-GAN optimization completed successfully!")

[I 2025-12-16 16:57:07,499] A new study created in memory with name: no-name-18a6bc5d-b5c8-49cd-aab5-bad5b7fba4d9


🔄 Reloading clean subset data for CTAB-GAN optimization...
✅ Clean data loaded: 2500 rows, 11 columns
✅ Missing values: 0
✅ Data validation passed: 0 missing values confirmed

🔧 SECTION 4.2: CTAB-GAN HYPERPARAMETER OPTIMIZATION
🔄 Creating CTAB-GAN optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2


🔄 CTAB-GAN Trial 1: epochs=300, batch_size=64, test_ratio=0.200


100%|██████████| 300/300 [07:50<00:00,  1.57s/it]


Finished training in 472.35508131980896  seconds.
🏋️ Training CTAB-GAN with corrected parameters...
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5439


[I 2025-12-16 17:05:00,635] Trial 0 finished with value: 0.6045595689011674 and parameters: {'epochs': 300, 'batch_size': 64, 'test_ratio': 0.2}. Best is trial 0 with value: 0.6045595689011674.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.6767
[CHART] Combined Score: 0.6110 (Similarity: 0.5671, Accuracy: 0.6767)
🎯 Trial 4 Results:
   • Combined Score: 0.6110
   • Similarity: 0.5671
   • Accuracy: 0.6767

🔄 CTAB-GAN Trial 5: epochs=500, batch_size=256, test_ratio=0.150


100%|██████████| 500/500 [13:06<00:00,  1.57s/it]


Finished training in 788.0196361541748  seconds.
🏋️ Training CTAB-GAN with corrected parameters...
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5530


[I 2025-12-16 17:40:32,957] Trial 4 finished with value: 0.6146939840251671 and parameters: {'epochs': 500, 'batch_size': 256, 'test_ratio': 0.15}. Best is trial 2 with value: 0.6196241078352104.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.7073
[CHART] Combined Score: 0.6147 (Similarity: 0.5530, Accuracy: 0.7073)
🎯 Trial 5 Results:
   • Combined Score: 0.6147
   • Similarity: 0.5530
   • Accuracy: 0.7073

✅ CTAB-GAN Optimization completed!
🏆 Best score: 0.6196
🔧 Best parameters:
   • epochs: 250
   • batch_size: 64
   • test_ratio: 0.25
✅ CTAB-GAN optimization completed successfully!


#### 4.1.3 CTAB-GAN+ Hyperparameter Optimization

In [31]:
# Code Chunk ID: CHUNK_4_1_3_A
# Import required libraries for CTAB-GAN+ optimization
import optuna
import numpy as np
import pandas as pd
from src.models.model_factory import ModelFactory
from src.evaluation.trts_framework import TRTSEvaluator

# ============================================================================
# CRITICAL FIX: Ensure clean, imputed subset data is loaded for CTAB-GAN+
# ============================================================================
print("🔄 Reloading clean subset data for CTAB-GAN+ optimization...")
data = pd.read_csv("data/liver_train_final_processed.csv")
print(f"✅ Clean data loaded: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"✅ Missing values: {data.isnull().sum().sum()}")

# Validate data quality
if data.isnull().sum().sum() > 0:
    raise ValueError("ERROR: CTAB-GAN+ data still contains missing values!")
else:
    print("✅ Data validation passed: 0 missing values confirmed")

# CORRECTED CTAB-GAN+ Search Space (3 supported parameters only)
def ctabganplus_search_space(trial):
    """Realistic CTAB-GAN+ hyperparameter space - ONLY supported parameters"""
    return {
        'epochs': trial.suggest_int('epochs', 150, 1000, step=50),
        'batch_size': trial.suggest_categorical('batch_size', [64, 128, 256]),  # Remove 500 - not stable
        'test_ratio': trial.suggest_float('test_ratio', 0.15, 0.25, step=0.05),
        # REMOVED: class_dim, random_dim, num_channels (not supported by constructor)
    }

def ctabganplus_objective(trial):
    """FINAL CORRECTED CTAB-GAN+ objective function - SAME PATTERN AS CTGAN"""
    try:
        # Get realistic hyperparameters from trial
        params = ctabganplus_search_space(trial)
        
        print(f"\n🔄 CTAB-GAN+ Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}, test_ratio={params['test_ratio']:.3f}")
        
        # Initialize CTAB-GAN+ using ModelFactory (SAME AS CTGAN)
        model = ModelFactory.create("ctabganplus", random_state=42)
        
        # CRITICAL FIX: Auto-detect categorical columns EXACTLY like CTGAN
        categorical_columns = data.select_dtypes(include=['object']).columns.tolist()
        print(f"🔍 Detected categorical columns: {categorical_columns}")
        
        # Train model with categorical columns (SAME PATTERN AS CTGAN)
        result = model.train(data, 
                           categorical_columns=categorical_columns,
                           epochs=params['epochs'],
                           batch_size=params['batch_size'],
                           test_ratio=params['test_ratio'])

        print(f"🏋️ Training CTAB-GAN+ with corrected parameters...")

        # Generate synthetic data for evaluation
        synthetic_data = model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0

# Create and run optimization study
print(f"\n🔧 SECTION 4.3: CTAB-GAN+ HYPERPARAMETER OPTIMIZATION")
print("=" * 80)
print(f"🔄 Creating CTAB-GAN+ optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

# Create study and optimize
ctabganplus_study = optuna.create_study(direction='maximize')
ctabganplus_study.optimize(ctabganplus_objective, n_trials=5)

# Extract and display results
best_trial = ctabganplus_study.best_trial
print(f"\n✅ CTAB-GAN+ Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ CTAB-GAN+ optimization completed successfully!")

[I 2025-12-16 17:40:32,999] A new study created in memory with name: no-name-0385cf7f-0d1d-4c5b-8965-368e391c296d


🔄 Reloading clean subset data for CTAB-GAN+ optimization...
✅ Clean data loaded: 2500 rows, 11 columns
✅ Missing values: 0
✅ Data validation passed: 0 missing values confirmed

🔧 SECTION 4.3: CTAB-GAN+ HYPERPARAMETER OPTIMIZATION
🔄 Creating CTAB-GAN+ optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2


🔄 CTAB-GAN+ Trial 1: epochs=350, batch_size=256, test_ratio=0.200
🔍 Detected categorical columns: []


100%|██████████| 350/350 [08:47<00:00,  1.51s/it]


Finished training in 529.959691286087  seconds.
🏋️ Training CTAB-GAN+ with corrected parameters...
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5606


[I 2025-12-16 17:49:23,764] Trial 0 finished with value: 0.6278354038594198 and parameters: {'epochs': 350, 'batch_size': 256, 'test_ratio': 0.2}. Best is trial 0 with value: 0.6278354038594198.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.7140
[CHART] Combined Score: 0.6257 (Similarity: 0.5668, Accuracy: 0.7140)
🎯 Trial 4 Results:
   • Combined Score: 0.6257
   • Similarity: 0.5668
   • Accuracy: 0.7140

🔄 CTAB-GAN+ Trial 5: epochs=700, batch_size=128, test_ratio=0.150
🔍 Detected categorical columns: []


100%|██████████| 700/700 [31:24<00:00,  2.69s/it]


Finished training in 1887.0496952533722  seconds.
🏋️ Training CTAB-GAN+ with corrected parameters...
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5540


[I 2025-12-16 19:12:11,196] Trial 4 finished with value: 0.6219728084796586 and parameters: {'epochs': 700, 'batch_size': 128, 'test_ratio': 0.15}. Best is trial 2 with value: 0.6384081788058877.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.7239
[CHART] Combined Score: 0.6220 (Similarity: 0.5540, Accuracy: 0.7239)
🎯 Trial 5 Results:
   • Combined Score: 0.6220
   • Similarity: 0.5540
   • Accuracy: 0.7239

✅ CTAB-GAN+ Optimization completed!
🏆 Best score: 0.6384
🔧 Best parameters:
   • epochs: 400
   • batch_size: 256
   • test_ratio: 0.2
✅ CTAB-GAN+ optimization completed successfully!


#### 4.1.4 GANerAid Hyperparameter Optimization

In [32]:
# Code Chunk ID: CHUNK_4_1_4_A
# GANerAid Search Space and Hyperparameter Optimization

# ============================================================================
# CRITICAL FIX: Ensure clean, imputed subset data is loaded for GANerAid
# ============================================================================
print("🔄 Reloading clean subset data for GANerAid optimization...")
import pandas as pd
data = pd.read_csv("data/liver_train_final_processed.csv")
print(f"✅ Clean data loaded: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"✅ Missing values: {data.isnull().sum().sum()}")

# Validate data quality
if data.isnull().sum().sum() > 0:
    raise ValueError("ERROR: GANerAid data still contains missing values!")
else:
    print("✅ Data validation passed: 0 missing values confirmed")

def ganeraid_search_space(trial):
    """
    GENERALIZED GANerAid hyperparameter search space with dynamic constraint adjustment.
    
    CRITICAL INSIGHT: Following CTGAN's compatible_pac pattern for robust constraint handling.
    GANerAid requires: batch_size % nr_of_rows == 0 AND nr_of_rows < dataset_size
    """
    
    # Define available batch sizes (easily extensible like CTGAN)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
    
    # Define dataset size constraint (GANerAid specific)
    dataset_size = len(data)  # Use current dataset size
    
    # Find compatible nr_of_rows values (same pattern as compatible_pac)
    max_nr_of_rows = min(dataset_size - 1, 500)  # Prevent index out of bounds
    possible_nr_of_rows = []
    
    # Find all compatible values (batch_size % nr_of_rows == 0)
    for candidate in range(1, max_nr_of_rows + 1):
        if batch_size % candidate == 0:
            possible_nr_of_rows.append(candidate)
    
    # Select nr_of_rows from compatible values
    if possible_nr_of_rows:
        nr_of_rows = trial.suggest_categorical(f'nr_of_rows_for_batch_{batch_size}', possible_nr_of_rows)
    else:
        # Fallback: use largest divisor of batch_size that's < dataset_size
        for candidate in range(batch_size, 0, -1):
            if batch_size % candidate == 0 and candidate < dataset_size:
                nr_of_rows = candidate
                break
        else:
            nr_of_rows = 1  # Ultimate fallback
    
    return {
        'epochs': trial.suggest_int('epochs', 500, 1500, step=100),
        'batch_size': batch_size,
        'nr_of_rows': nr_of_rows,
    }

def ganeraid_objective(trial):
    """GENERALIZED GANerAid objective function with ALL constraint validation."""
    try:
        # Get hyperparameters from trial
        params = ganeraid_search_space(trial)
        
        # DYNAMIC CONSTRAINT ADJUSTMENT (following CTGAN pattern)
        dataset_size = len(data)
        batch_size = params['batch_size']
        original_nr_of_rows = params['nr_of_rows']
        
        # Comprehensive constraint validation
        compatible_nr_of_rows = original_nr_of_rows
        found_compatible = False
        
        # Try to find compatible nr_of_rows (batch_size % nr_of_rows == 0 AND nr_of_rows < dataset_size)
        for candidate in range(original_nr_of_rows, 0, -1):
            if (batch_size % candidate == 0 and 
                candidate < dataset_size):
                compatible_nr_of_rows = candidate
                found_compatible = True
                break
        
        # If still not compatible, try upward
        if not found_compatible:
            for candidate in range(original_nr_of_rows + 1, min(dataset_size, batch_size + 1)):
                if (batch_size % candidate == 0 and 
                    candidate < dataset_size):
                    compatible_nr_of_rows = candidate
                    found_compatible = True
                    break
        
        # Ultimate fallback
        if not found_compatible:
            compatible_nr_of_rows = 1
        
        params['nr_of_rows'] = compatible_nr_of_rows
        
        print(f"\\n🔄 GANerAid Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}, nr_of_rows={params['nr_of_rows']}")
        print(f"✅ Constraint validation: {batch_size} % {compatible_nr_of_rows} = {batch_size % compatible_nr_of_rows}, {compatible_nr_of_rows} < {dataset_size}")

        # Initialize GANerAid using ModelFactory
        from src.models.model_factory import ModelFactory
        model = ModelFactory.create("ganeraid", random_state=42)
        
        # Train model
        print(f"🏋️ Training GANerAid with validated parameters...")
        start_time = time.time()
        
        try:
            model.train(data, epochs=params['epochs'])
            training_time = time.time() - start_time
            print(f"⏱️ Training completed successfully in {training_time:.1f} seconds")
        except IndexError as e:
            print(f"❌ IndexError during training (constraint violation): {str(e)}")
            return 0.0
        except Exception as e:
            print(f"❌ Training failed: {str(e)}")
            return 0.0

        # Generate synthetic data for evaluation
        synthetic_data = model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0

# Create and run optimization study
import optuna
import time

print(f"\\n🔧 SECTION 4.4: GANerAid HYPERPARAMETER OPTIMIZATION")
print("=" * 80)
print(f"🔄 Creating GANerAid optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

# Create study and optimize
ganeraid_study = optuna.create_study(direction='maximize')
ganeraid_study.optimize(ganeraid_objective, n_trials=5)

# Extract and display results
best_trial = ganeraid_study.best_trial
print(f"\\n✅ GANerAid Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ GANerAid optimization completed successfully!")

[I 2025-12-16 19:12:11,222] A new study created in memory with name: no-name-1774a35d-1252-4db4-8175-caa06ef76da8


🔄 Reloading clean subset data for GANerAid optimization...
✅ Clean data loaded: 2500 rows, 11 columns
✅ Missing values: 0
✅ Data validation passed: 0 missing values confirmed
\n🔧 SECTION 4.4: GANerAid HYPERPARAMETER OPTIMIZATION
🔄 Creating GANerAid optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2

\n🔄 GANerAid Trial 1: epochs=1500, batch_size=256, nr_of_rows=8
✅ Constraint validation: 256 % 8 = 0, 8 < 2500
🏋️ Training GANerAid with validated parameters...
Initialized gan with the following parameters: 
lr_d = 0.0005
lr_g = 0.0005
hidden_feature_space = 200
batch_size = 100
nr_of_rows = 25
binary_noise = 0.2
Start training of gan for 1500 epochs


100%|██████████| 1500/1500 [06:47<00:00,  3.68it/s, loss=d error: 1.0706804990768433 --- g error 1.4602134227752686] 


⏱️ Training completed successfully in 408.0 seconds
Generating 5000 samples
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.4559


[I 2025-12-16 19:19:00,788] Trial 0 finished with value: 0.5493666624666631 and parameters: {'batch_size': 256, 'nr_of_rows_for_batch_256': 8, 'epochs': 1500}. Best is trial 0 with value: 0.5493666624666631.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.6702
[CHART] Combined Score: 0.5435 (Similarity: 0.4590, Accuracy: 0.6702)
🎯 Trial 4 Results:
   • Combined Score: 0.5435
   • Similarity: 0.4590
   • Accuracy: 0.6702
\n🔄 GANerAid Trial 5: epochs=1300, batch_size=128, nr_of_rows=1
✅ Constraint validation: 128 % 1 = 0, 1 < 2500
🏋️ Training GANerAid with validated parameters...
Initialized gan with the following parameters: 
lr_d = 0.0005
lr_g = 0.0005
hidden_feature_space = 200
batch_size = 100
nr_of_rows = 25
binary_noise = 0.2
Start training of gan for 1300 epochs


100%|██████████| 1300/1300 [05:18<00:00,  4.08it/s, loss=d error: 0.8666309714317322 --- g error 2.756155014038086]  


⏱️ Training completed successfully in 318.7 seconds
Generating 5000 samples
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5387


[I 2025-12-16 19:33:35,938] Trial 4 finished with value: 0.6006387600774246 and parameters: {'batch_size': 128, 'nr_of_rows_for_batch_128': 1, 'epochs': 1300}. Best is trial 4 with value: 0.6006387600774246.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6935
[CHART] Combined Score: 0.6006 (Similarity: 0.5387, Accuracy: 0.6935)
🎯 Trial 5 Results:
   • Combined Score: 0.6006
   • Similarity: 0.5387
   • Accuracy: 0.6935
\n✅ GANerAid Optimization completed!
🏆 Best score: 0.6006
🔧 Best parameters:
   • batch_size: 128
   • nr_of_rows_for_batch_128: 1
   • epochs: 1300
✅ GANerAid optimization completed successfully!


#### 4.1.5 CopulaGAN Hyperparameter Optimization

In [33]:
# Code Chunk ID: CHUNK_4_1_5_A
# CopulaGAN Search Space and Hyperparameter Optimization

# ============================================================================
# CRITICAL FIX: Ensure clean, imputed subset data is loaded for CopulaGAN
# ============================================================================
print("🔄 Reloading clean subset data for CopulaGAN optimization...")
import pandas as pd
data = pd.read_csv("data/liver_train_final_processed.csv")
print(f"✅ Clean data loaded: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"✅ Missing values: {data.isnull().sum().sum()}")

# Validate data quality
if data.isnull().sum().sum() > 0:
    raise ValueError("ERROR: CopulaGAN data still contains missing values!")
else:
    print("✅ Data validation passed: 0 missing values confirmed")

def copulagan_search_space(trial):
    """
    GENERALIZED CopulaGAN hyperparameter search space with dynamic constraint adjustment.
    
    CRITICAL INSIGHT: Following CTGAN's compatible_pac pattern for robust constraint handling.
    CopulaGAN requires discrete_columns to be properly defined.
    """
    return {
        'epochs': trial.suggest_int('epochs', 50, 500, step=50),
        'batch_size': trial.suggest_categorical('batch_size', [100, 200, 500]),
        'generator_lr': trial.suggest_loguniform('generator_lr', 1e-5, 1e-2),
        'discriminator_lr': trial.suggest_loguniform('discriminator_lr', 1e-5, 1e-2),
        'generator_decay': trial.suggest_loguniform('generator_decay', 1e-8, 1e-4),
        'discriminator_decay': trial.suggest_loguniform('discriminator_decay', 1e-8, 1e-4),
    }

def copulagan_objective(trial):
    """GENERALIZED CopulaGAN objective function."""
    try:
        # Get hyperparameters from trial
        params = copulagan_search_space(trial)
        
        print(f"\\n🔄 CopulaGAN Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}")
        
        # Initialize CopulaGAN using ModelFactory
        from src.models.model_factory import ModelFactory
        model = ModelFactory.create("copulagan", random_state=42)
        
        # Auto-detect discrete columns for CopulaGAN
        discrete_columns = data.select_dtypes(include=['object']).columns.tolist()
        print(f"📊 Detected discrete columns: {discrete_columns}")
        
        # Train model
        print(f"🏋️ Training CopulaGAN...")
        start_time = time.time()
        
        try:
            model.train(data, discrete_columns=discrete_columns, **params)
            training_time = time.time() - start_time
            print(f"⏱️ Training completed successfully in {training_time:.1f} seconds")
        except Exception as e:
            print(f"❌ Training failed: {str(e)}")
            return 0.0

        # Generate synthetic data for evaluation
        synthetic_data = model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0

# Create and run optimization study
import optuna
import time

print(f"\\n🔧 SECTION 4.5: CopulaGAN HYPERPARAMETER OPTIMIZATION")
print("=" * 80)
print(f"🔄 Creating CopulaGAN optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

# Create study and optimize
copulagan_study = optuna.create_study(direction='maximize')
copulagan_study.optimize(copulagan_objective, n_trials=5)

# Extract and display results
best_trial = copulagan_study.best_trial
print(f"\\n✅ CopulaGAN Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ CopulaGAN optimization completed successfully!")

[I 2025-12-16 19:33:35,962] A new study created in memory with name: no-name-51e2b8df-ec77-4782-b069-1c9c37ad1cc1


🔄 Reloading clean subset data for CopulaGAN optimization...
✅ Clean data loaded: 2500 rows, 11 columns
✅ Missing values: 0
✅ Data validation passed: 0 missing values confirmed
\n🔧 SECTION 4.5: CopulaGAN HYPERPARAMETER OPTIMIZATION
🔄 Creating CopulaGAN optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2

\n🔄 CopulaGAN Trial 1: epochs=300, batch_size=100
📊 Detected discrete columns: []
🏋️ Training CopulaGAN...
⏱️ Training completed successfully in 135.5 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5034


[I 2025-12-16 19:35:52,561] Trial 0 finished with value: 0.5817891711589351 and parameters: {'epochs': 300, 'batch_size': 100, 'generator_lr': 0.00028584360651275715, 'discriminator_lr': 0.00036895113248203214, 'generator_decay': 5.697396771771347e-08, 'discriminator_decay': 3.262268031937437e-06}. Best is trial 0 with value: 0.5817891711589351.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6993
[CHART] Combined Score: 0.5818 (Similarity: 0.5034, Accuracy: 0.6993)
🎯 Trial 1 Results:
   • Combined Score: 0.5818
   • Similarity: 0.5034
   • Accuracy: 0.6993
\n🔄 CopulaGAN Trial 2: epochs=500, batch_size=100
📊 Detected discrete columns: []
🏋️ Training CopulaGAN...
⏱️ Training completed successfully in 216.8 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5864


[I 2025-12-16 19:39:30,464] Trial 1 finished with value: 0.6108239484614009 and parameters: {'epochs': 500, 'batch_size': 100, 'generator_lr': 3.266153568308089e-05, 'discriminator_lr': 0.0011205396890051405, 'generator_decay': 7.496598360108174e-05, 'discriminator_decay': 1.3067663019475162e-07}. Best is trial 1 with value: 0.6108239484614009.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.6390
[CHART] Combined Score: 0.5785 (Similarity: 0.5382, Accuracy: 0.6390)
🎯 Trial 3 Results:
   • Combined Score: 0.5785
   • Similarity: 0.5382
   • Accuracy: 0.6390
\n🔄 CopulaGAN Trial 4: epochs=400, batch_size=500
📊 Detected discrete columns: []
🏋️ Training CopulaGAN...
⏱️ Training completed successfully in 69.4 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5784


[I 2025-12-16 19:41:34,647] Trial 3 finished with value: 0.6233569200350036 and parameters: {'epochs': 400, 'batch_size': 500, 'generator_lr': 0.0028960879454961222, 'discriminator_lr': 0.00837270747510529, 'generator_decay': 8.4888660578074e-05, 'discriminator_decay': 4.343040428293054e-08}. Best is trial 3 with value: 0.6233569200350036.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6908
[CHART] Combined Score: 0.6234 (Similarity: 0.5784, Accuracy: 0.6908)
🎯 Trial 4 Results:
   • Combined Score: 0.6234
   • Similarity: 0.5784
   • Accuracy: 0.6908
\n🔄 CopulaGAN Trial 5: epochs=50, batch_size=100
📊 Detected discrete columns: []
🏋️ Training CopulaGAN...
⏱️ Training completed successfully in 27.9 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5187


[I 2025-12-16 19:42:03,576] Trial 4 finished with value: 0.576766422028639 and parameters: {'epochs': 50, 'batch_size': 100, 'generator_lr': 0.00031505509717684937, 'discriminator_lr': 0.0010228958562560648, 'generator_decay': 1.630201350408089e-06, 'discriminator_decay': 2.6532672784287268e-05}. Best is trial 3 with value: 0.6233569200350036.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6639
[CHART] Combined Score: 0.5768 (Similarity: 0.5187, Accuracy: 0.6639)
🎯 Trial 5 Results:
   • Combined Score: 0.5768
   • Similarity: 0.5187
   • Accuracy: 0.6639
\n✅ CopulaGAN Optimization completed!
🏆 Best score: 0.6234
🔧 Best parameters:
   • epochs: 400
   • batch_size: 500
   • generator_lr: 0.0028960879454961222
   • discriminator_lr: 0.00837270747510529
   • generator_decay: 8.4888660578074e-05
   • discriminator_decay: 4.343040428293054e-08
✅ CopulaGAN optimization completed successfully!


#### 4.1.6 TVAE Hyperparameter Optimization

In [34]:
# Code Chunk ID: CHUNK_4_1_6_A
# TVAE Robust Search Space (from hypertuning_eg.md)

# ============================================================================
# CRITICAL FIX: Ensure clean, imputed subset data is loaded for TVAE
# ============================================================================
print("🔄 Reloading clean subset data for TVAE optimization...")
import pandas as pd
data = pd.read_csv("data/liver_train_final_processed.csv")
print(f"✅ Clean data loaded: {data.shape[0]} rows, {data.shape[1]} columns")
print(f"✅ Missing values: {data.isnull().sum().sum()}")

# Validate data quality
if data.isnull().sum().sum() > 0:
    raise ValueError("ERROR: TVAE data still contains missing values!")
else:
    print("✅ Data validation passed: 0 missing values confirmed")

def tvae_search_space(trial):
    return {
        "epochs": trial.suggest_int("epochs", 50, 500, step=50),  # Training cycles
        "batch_size": trial.suggest_categorical("batch_size", [100, 200, 500]),  # Batch size for training
        "embedding_dim": trial.suggest_categorical("embedding_dim", [64, 128]),  # Embedding dimension
        "compress_dims": trial.suggest_categorical("compress_dims", [(128, 128), (256, 256)]),  # Compression layers
        "decompress_dims": trial.suggest_categorical("decompress_dims", [(128, 128), (256, 256)]),  # Decompression layers
        "l2scale": trial.suggest_loguniform("l2scale", 1e-8, 1e-3),  # L2 regularization
        "loss_factor": trial.suggest_int("loss_factor", 1, 5),  # Loss scaling factor
    }

def tvae_objective(trial):
    """TVAE objective function with comprehensive error handling."""
    try:
        # Get hyperparameters from trial
        params = tvae_search_space(trial)
        
        print(f"\\n🔄 TVAE Trial {trial.number + 1}: epochs={params['epochs']}, batch_size={params['batch_size']}, embedding_dim={params['embedding_dim']}")
        
        # Initialize TVAE using ModelFactory
        from src.models.model_factory import ModelFactory
        model = ModelFactory.create("tvae", random_state=42)
        
        # Train model
        print("🏋️ Training TVAE...")
        start_time = time.time()
        model.train(data, **params)
        training_time = time.time() - start_time
        print(f"⏱️ Training completed in {training_time:.1f} seconds")
        
        # Generate synthetic data for evaluation
        synthetic_data = model.generate(5000)
        print(f"📊 Generated synthetic data: {synthetic_data.shape}")
        
        # Use enhanced objective function
        from setup import enhanced_objective_function_v2
        combined_score, similarity_score, accuracy_score = enhanced_objective_function_v2(
            data, synthetic_data, TARGET_COLUMN
        )
        
        print(f"🎯 Trial {trial.number + 1} Results:")
        print(f"   • Combined Score: {combined_score:.4f}")
        print(f"   • Similarity: {similarity_score:.4f}")
        print(f"   • Accuracy: {accuracy_score:.4f}")
        
        return combined_score
        
    except Exception as e:
        print(f"❌ Trial {trial.number + 1} failed: {str(e)}")
        import traceback
        traceback.print_exc()
        return 0.0

# Create and run optimization study
import optuna
import time

print(f"\\n🔧 SECTION 4.6: TVAE HYPERPARAMETER OPTIMIZATION")
print("=" * 80)
print(f"🔄 Creating TVAE optimization study...")
print(f"📊 Dataset info: {len(data)} rows, {len(data.columns)} columns")
print(f"📊 Target column '{TARGET_COLUMN}' unique values: {data[TARGET_COLUMN].nunique()}")
print()

# Create study and optimize
tvae_study = optuna.create_study(direction='maximize')
tvae_study.optimize(tvae_objective, n_trials=5)

# Extract and display results
best_trial = tvae_study.best_trial
print(f"\\n✅ TVAE Optimization completed!")
print(f"🏆 Best score: {best_trial.value:.4f}")
print(f"🔧 Best parameters:")
for param, value in best_trial.params.items():
    print(f"   • {param}: {value}")

print("✅ TVAE optimization completed successfully!")

[I 2025-12-16 19:42:03,601] A new study created in memory with name: no-name-bcf81a3d-ff68-4e17-9174-5c2e2e394934


🔄 Reloading clean subset data for TVAE optimization...
✅ Clean data loaded: 2500 rows, 11 columns
✅ Missing values: 0
✅ Data validation passed: 0 missing values confirmed
\n🔧 SECTION 4.6: TVAE HYPERPARAMETER OPTIMIZATION
🔄 Creating TVAE optimization study...
📊 Dataset info: 2500 rows, 11 columns
📊 Target column 'Result' unique values: 2

\n🔄 TVAE Trial 1: epochs=450, batch_size=200, embedding_dim=128
🏋️ Training TVAE...
⏱️ Training completed in 29.3 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5585


[I 2025-12-16 19:42:33,687] Trial 0 finished with value: 0.6335833107889015 and parameters: {'epochs': 450, 'batch_size': 200, 'embedding_dim': 128, 'compress_dims': (128, 128), 'decompress_dims': (256, 256), 'l2scale': 5.159492492706098e-07, 'loss_factor': 4}. Best is trial 0 with value: 0.6335833107889015.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.7462
[CHART] Combined Score: 0.6336 (Similarity: 0.5585, Accuracy: 0.7462)
🎯 Trial 1 Results:
   • Combined Score: 0.6336
   • Similarity: 0.5585
   • Accuracy: 0.7462
\n🔄 TVAE Trial 2: epochs=150, batch_size=200, embedding_dim=64
🏋️ Training TVAE...
⏱️ Training completed in 11.1 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.4847


[I 2025-12-16 19:42:45,472] Trial 1 finished with value: 0.5819135402794331 and parameters: {'epochs': 150, 'batch_size': 200, 'embedding_dim': 64, 'compress_dims': (256, 256), 'decompress_dims': (256, 256), 'l2scale': 3.4714702767884368e-06, 'loss_factor': 2}. Best is trial 0 with value: 0.6335833107889015.


...

[OK] TRTS Evaluation: 2 scenarios, Average: 0.7291
[CHART] Combined Score: 0.6023 (Similarity: 0.5178, Accuracy: 0.7291)
🎯 Trial 3 Results:
   • Combined Score: 0.6023
   • Similarity: 0.5178
   • Accuracy: 0.7291
\n🔄 TVAE Trial 4: epochs=300, batch_size=200, embedding_dim=64
🏋️ Training TVAE...
⏱️ Training completed in 19.9 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5741


[I 2025-12-16 19:43:18,023] Trial 3 finished with value: 0.6467980619457332 and parameters: {'epochs': 300, 'batch_size': 200, 'embedding_dim': 64, 'compress_dims': (128, 128), 'decompress_dims': (256, 256), 'l2scale': 1.065933663599726e-08, 'loss_factor': 4}. Best is trial 3 with value: 0.6467980619457332.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.7558
[CHART] Combined Score: 0.6468 (Similarity: 0.5741, Accuracy: 0.7558)
🎯 Trial 4 Results:
   • Combined Score: 0.6468
   • Similarity: 0.5741
   • Accuracy: 0.7558
\n🔄 TVAE Trial 5: epochs=50, batch_size=100, embedding_dim=64
🏋️ Training TVAE...
⏱️ Training completed in 4.8 seconds
📊 Generated synthetic data: (5000, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.4201


[I 2025-12-16 19:43:23,647] Trial 4 finished with value: 0.5239169415313205 and parameters: {'epochs': 50, 'batch_size': 100, 'embedding_dim': 64, 'compress_dims': (256, 256), 'decompress_dims': (256, 256), 'l2scale': 0.0002267674126088336, 'loss_factor': 3}. Best is trial 3 with value: 0.6467980619457332.


[OK] TRTS Evaluation: 2 scenarios, Average: 0.6796
[CHART] Combined Score: 0.5239 (Similarity: 0.4201, Accuracy: 0.6796)
🎯 Trial 5 Results:
   • Combined Score: 0.5239
   • Similarity: 0.4201
   • Accuracy: 0.6796
\n✅ TVAE Optimization completed!
🏆 Best score: 0.6468
🔧 Best parameters:
   • epochs: 300
   • batch_size: 200
   • embedding_dim: 64
   • compress_dims: (128, 128)
   • decompress_dims: (256, 256)
   • l2scale: 1.065933663599726e-08
   • loss_factor: 4
✅ TVAE optimization completed successfully!


In [35]:
# Create Optuna summary comparing all models
from src.visualization.section4 import create_all_models_optuna_summary

# Collect all studies (only include those that completed successfully)
studies_dict = {}
if 'ctgan_study' in locals():
    studies_dict['CTGAN'] = ctgan_study
if 'ctabgan_study' in locals():
    studies_dict['CTABGAN'] = ctabgan_study
if 'ctabganplus_study' in locals():
    studies_dict['CTABGAN+'] = ctabganplus_study
if 'ganeraid_study' in locals():
    studies_dict['GANerAid'] = ganeraid_study
if 'copulagan_study' in locals():
    studies_dict['CopulaGAN'] = copulagan_study
if 'tvae_study' in locals():
    studies_dict['TVAE'] = tvae_study

if studies_dict:
    create_all_models_optuna_summary(
        studies_dict=studies_dict,
        results_path=results_path,
        verbose=True
    )
    print(f"\n[OK] Optuna summary visualization created for {len(studies_dict)} models")
else:
    print("[WARNING] No Optuna studies available for summary visualization")


[VIZ] Saved: optuna_summary_all_models.png

[OK] Optuna summary visualization created for 6 models


### 4.2 Batch process 

This section outputs figures and graphics from models run in 4.1. The next chunk will output results for whatever subsections of 4.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 4.1.

In [36]:
# Code Chunk ID: CHUNK_4_2_0_A
# ============================================================================
# SECTION 4 - BATCH HYPERPARAMETER OPTIMIZATION ANALYSIS
# ============================================================================

print("🔍 SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS")
print("=" * 80)
print()

# Use enhanced batch evaluation function from setup.py
# Following exact same pattern as CHUNK_018 (Section 3) - no module reload needed!
try:
    # Run batch analysis with file export for all models
    section4_batch_results = evaluate_hyperparameter_optimization_results(
        section_number=4,
        scope=globals(),  # Pass notebook scope to access study variables
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("✅ SECTION 4 HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS COMPLETED!")
    print("="*80)
    print(f"📊 Models processed: {len(section4_batch_results['summary_data'])}")
    print(f"📁 Results exported to: {section4_batch_results['results_dir']}")
    print(f"📋 Individual model analysis files:")
    print("   • Hyperparameter parameter_analysis.png plots")
    print("   • Optimization convergence_analysis.png graphs")
    print("   • Parameter correlation matrices")
    print("   • Best trial summary tables")
    print("   • Comprehensive optimization summary CSV")

    
except Exception as e:
    print(f"❌ Batch hyperparameter analysis failed: {str(e)}")
    print(f"🔍 Error details: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    print("\n⚠️  Falling back to individual chunk analysis if needed")

# ============================================================================
# SAVE BEST PARAMETERS TO CSV FOR SECTION 5 USE
# ============================================================================
print("\n" + "=" * 80)
print("💾 SAVING BEST PARAMETERS FROM SECTION 4 OPTIMIZATION")
print("=" * 80)

try:
    # Save all best parameters to CSV using setup.py function
    param_save_results = save_best_parameters_to_csv(
        scope=globals(),
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER
    )
    
    if param_save_results['success']:
        print(f"\n✅ Parameter saving completed successfully!")
        print(f"   • Files saved: {len(param_save_results['files_saved'])}")
        print(f"   • Parameter entries: {param_save_results['parameters_count']}")
        print(f"   • Models processed: {param_save_results['models_count']}")
        print(f"   • Directory: {param_save_results['results_dir']}")
        
        # Display saved files
        for file_path in param_save_results['files_saved']:
            print(f"     📁 {file_path.split('/')[-1]}")
    else:
        print(f"\n⚠️  Parameter saving completed with issues: {param_save_results['message']}")
        
except Exception as e:
    print(f"\n❌ Parameter saving failed: {str(e)}")
    print(f"   Section 5 will fall back to memory-based parameter retrieval")

print(f"\n📈 Section 4 hyperparameter optimization analysis complete!")
print("🏁 Ready for Section 5: Optimized model re-training")

🔍 SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS

[LOCATION] Using DATASET_IDENTIFIER from scope: liver-train
[TARGET] Final DATASET_IDENTIFIER for Section 4: liver-train

SECTION 4 - HYPERPARAMETER OPTIMIZATION BATCH ANALYSIS
[FOLDER] Base results directory: results/liver-train/2025-12-16/Section-4
[TARGET] Target column: Result
[CHART] Dataset identifier: liver-train


[SEARCH] 4.1.1: CTGAN Hyperparameter Optimization Analysis
------------------------------------------------------------
[OK] CTGAN optimization study found
[FOLDER] Model directory: results/liver-train/2025-12-16/Section-4/CTGAN
[SEARCH] ANALYZING CTGAN HYPERPARAMETER OPTIMIZATION
[CHART] 1. TRIAL DATA EXTRACTION AND PROCESSING
----------------------------------------
[OK] Extracted 100 trials for analysis
[CHART] 2. PARAMETER SPACE EXPLORATION ANALYSIS
----------------------------------------
   - Found 12 hyperparameters: ['params_batch_size', 'params_discriminator_decay', 'params_discriminator_dim', 'params_

## Section 5: Final Model Comparison and Best-of-Best Selection

### 5.1 Re-run of models with best hyperparameters identified from Section 4

#### 5.1.1 Best CTGAN Model Evaluation

In [37]:
# Code Chunk ID: CHUNK_5_1_1_A
# Section 5.1: Best CTGAN Model Evaluation  
print("🏆 SECTION 5.1: BEST CTGAN MODEL EVALUATION")
print("=" * 60)

# ============================================================================
# LOAD BEST PARAMETERS FROM SECTION 4 (CSV + MEMORY FALLBACK)
# ============================================================================
print("📖 5.1.0 Loading best parameters from Section 4...")

try:
    # Load all best parameters using setup.py function
    param_data = load_best_parameters_from_csv(
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        fallback_to_memory=True,
        scope=globals()
    )
    
    print(f"✅ Parameter loading completed from {param_data['source']}")
    print(f"   • Models available: {param_data['models_count']}")
    
    # Extract CTGAN parameters specifically
    loaded_ctgan_params = param_data['parameters'].get('ctgan', None)
    
except Exception as e:
    print(f"⚠️  Parameter loading failed: {str(e)}")
    print(f"   Falling back to direct memory access")
    loaded_ctgan_params = None

# 5.1.1 Retrieve Best Model Results from Section 4.1
print("\n📊 5.1.1 Retrieving best CTGAN results from Section 4.1...")

try:
    # Primary: Use loaded parameters if available
    if loaded_ctgan_params is not None:
        print(f"✅ Using loaded CTGAN parameters from {param_data['source']}")
        best_params = loaded_ctgan_params
        
        # Try to get additional metadata from memory if available
        if 'ctgan_study' in globals() and ctgan_study is not None and hasattr(ctgan_study, 'best_trial'):
            best_trial = ctgan_study.best_trial
            best_value = best_trial.value
            trial_number = best_trial.number
        else:
            # Use fallback values when memory unavailable  
            best_value = 0.0  # Will be recalculated during evaluation
            trial_number = "loaded_from_csv"
            print(f"   ⚠️  Memory study unavailable - using loaded parameters only")
        
    else:
        # Fallback: Direct memory access
        print(f"🔄 Falling back to direct memory access...")
        best_trial = ctgan_study.best_trial
        best_params = best_trial.params
        best_value = best_trial.value
        trial_number = best_trial.number
        print(f"✅ Using CTGAN parameters from memory")
    
    print(f"\n✅ Section 4.1 CTGAN optimization parameters retrieved!")
    print(f"   • Best Trial: #{trial_number}")
    print(f"   • Best Objective Score: {best_value:.4f}" if isinstance(best_value, (int, float)) else f"   • Best Objective Score: {best_value}")
    print(f"   • Parameter count: {len(best_params)}")
    
    # Display parameters
    print(f"\n📈 5.1.2 Best CTGAN configuration:")
    for param, value in best_params.items():
        if isinstance(value, float):
            print(f"   • {param}: {value:.4f}")
        else:
            print(f"   • {param}: {value}")
    
    print(f"🔍 Parameter source: {param_data.get('source', 'memory') if loaded_ctgan_params else 'memory'}")
    
    # ============================================================================
    # 5.1.3 TRAIN FINAL CTGAN MODEL WITH OPTIMIZED PARAMETERS
    # ============================================================================
    
    print(f"\n🔧 5.1.3 Training final CTGAN model with optimized parameters...")
    
    try:
        # Use ModelFactory pattern
        from src.models.model_factory import ModelFactory
        
        # Create CTGAN model
        final_ctgan_model = ModelFactory.create("ctgan", random_state=42)
        
        # Apply best parameters with defaults for missing values
        final_ctgan_params = {
            'epochs': best_params.get('epochs', 300),
            'batch_size': best_params.get('batch_size', 500),
            'generator_lr': best_params.get('generator_lr', 2e-4),
            'discriminator_lr': best_params.get('discriminator_lr', 2e-4),
            'generator_decay': best_params.get('generator_decay', 1e-6),
            'discriminator_decay': best_params.get('discriminator_decay', 1e-6),
            'pac': best_params.get('pac', 10),
            'verbose': best_params.get('verbose', True)
        }
        
        print("🔧 Training CTGAN with optimal hyperparameters...")
        for param, value in final_ctgan_params.items():
            print(f"   • Using {param}: {value}")
        
        # Train the model
        final_ctgan_model.train(data, **final_ctgan_params)
        print("✅ CTGAN training completed successfully!")
        
        # Generate synthetic data
        print("🎲 Generating synthetic data...")
        synthetic_ctgan_final = final_ctgan_model.generate(len(data))
        print(f"✅ Generated {len(synthetic_ctgan_final)} synthetic samples")
        
        # ============================================================================
        # 5.1.4 EVALUATE FINAL CTGAN MODEL PERFORMANCE
        # ============================================================================
        
        print("\n📊 5.1.4 Final CTGAN Model Evaluation...")
        
        # Use enhanced objective function for evaluation
        if 'enhanced_objective_function_v2' in globals():
            print("🎯 Enhanced objective function evaluation:")
            
            ctgan_final_score, ctgan_similarity, ctgan_accuracy = enhanced_objective_function_v2(
                real_data=data, 
                synthetic_data=synthetic_ctgan_final, 
                target_column=TARGET_COLUMN
            )
            
            print(f"\n✅ Final CTGAN Evaluation Results:")
            print(f"   • Overall Score: {ctgan_final_score:.4f}")
            print(f"   • Similarity Score: {ctgan_similarity:.4f} (60% weight)")  
            print(f"   • Accuracy Score: {ctgan_accuracy:.4f} (40% weight)")
            
            # Store results for Section 5.7 comparison
            ctgan_final_results = {
                'model_name': 'CTGAN',
                'objective_score': ctgan_final_score,
                'similarity_score': ctgan_similarity,
                'accuracy_score': ctgan_accuracy,
                'best_params': best_params,
                'parameter_source': param_data.get('source', 'memory') if loaded_ctgan_params else 'memory',
                'synthetic_data': synthetic_ctgan_final
            }
            
            print("🎯 CTGAN Final Assessment:")
            print(f"   • Production Ready: {'✅ Yes' if ctgan_final_score > 0.6 else '⚠️ Review Required'}")
            print(f"   • Recommended for: General-purpose tabular synthetic data generation")
            print(f"   • Final Score vs Optimization Score: {ctgan_final_score:.4f} vs {best_value:.4f}" if isinstance(best_value, (int, float)) else f"   • Final Score: {ctgan_final_score:.4f}")
            
        else:
            print("⚠️ Enhanced objective function not available - using basic evaluation")
            ctgan_final_results = {
                'model_name': 'CTGAN',
                'objective_score': best_value if isinstance(best_value, (int, float)) else 0.0,
                'best_params': best_params,
                'parameter_source': param_data.get('source', 'memory') if loaded_ctgan_params else 'memory',
                'synthetic_data': synthetic_ctgan_final
            }
                
    except Exception as train_error:
        print(f"❌ Failed to train final CTGAN model: {train_error}")
        import traceback
        traceback.print_exc()
        synthetic_ctgan_final = None
        ctgan_final_score = 0.0
        ctgan_final_results = {
            'model_name': 'CTGAN',
            'objective_score': 0.0,
            'error': str(train_error)
        }

except Exception as e:
    print(f"❌ Error accessing CTGAN parameters: {e}")
    print("   Please ensure Section 4.1 has been executed successfully or parameter CSV exists.")
    # Create empty results to prevent downstream errors
    synthetic_ctgan_final = None
    ctgan_final_results = {
        'model_name': 'CTGAN',
        'objective_score': 0.0,
        'error': str(e)
    }
    
print("\n" + "=" * 60)
print("✅ SECTION 5.1 COMPLETE: Best CTGAN model trained and evaluated")
print("🔄 Ready for Section 5.2: CTAB-GAN model training")

🏆 SECTION 5.1: BEST CTGAN MODEL EVALUATION
📖 5.1.0 Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/liver-train/2025-12-16/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 12 parameters
[OK] Loaded CTAB-GAN: 3 parameters
[OK] Loaded CTAB-GAN+: 3 parameters
[OK] Loaded GANerAid: 3 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 7 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 12 parameters
   - ctabgan: 3 parameters
   - ctabganplus: 3 parameters
   - ganeraid: 3 parameters
   - copulagan: 6 parameters
   - tvae: 7 parameters
✅ Parameter loading completed from CSV file
   • Models available: 6

📊 5.1.1 Retrieving best CTGAN results from Section 4.1...
✅ Using loaded CTGAN parameters from CSV file

✅ Section 4.1 CTGAN optimization parameters retrieved!
   • Best Trial: #92
   • Best Objective Score: 0.6657
   •

Gen. (-0.99) | Discrim. (-0.33): 100%|██████████| 900/900 [02:30<00:00,  6.00it/s]


✅ CTGAN training completed successfully!
🎲 Generating synthetic data...
✅ Generated 2500 synthetic samples

📊 5.1.4 Final CTGAN Model Evaluation...
🎯 Enhanced objective function evaluation:
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5676
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6672
[CHART] Combined Score: 0.6074 (Similarity: 0.5676, Accuracy: 0.6672)

✅ Final CTGAN Evaluation Results:
   • Overall Score: 0.6074
   • Similarity Score: 0.5676 (60% weight)
   • Accuracy Score: 0.6672 (40% weight)
🎯 CTGAN Final Assessment:
   • Production Ready: ✅ Yes
   • Recommended for: General-purpose tabular synthetic data generation
   • Final Score vs Optimization Score: 0.6074 vs 0.6657

✅ SECTION 5.1 COMPLETE: Best CTGAN model trained and evaluated
🔄 Ready for Section 5.2: CTAB-GAN model training


#### 5.1.2 Best CTAB-GAN Model Evaluation

In [38]:
# Code Chunk ID: CHUNK_5_1_1_Aa

# Section 5.2: Best CTAB-GAN Model Evaluation
print("🏆 SECTION 5.2: BEST CTAB-GAN MODEL EVALUATION")
print("=" * 60)

# 5.2.1 Retrieve Best Model Results from Section 4.2
print("📊 5.2.1 Retrieving best CTAB-GAN results from Section 4.2...")

try:
    # Use unified parameter loading function
    ctabgan_params = get_model_parameters(
        model_name='ctab-gan',
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        scope=globals()
    )
    
    if ctabgan_params is not None:
        best_params = ctabgan_params
        
        # Try to get additional metadata from memory if available
        if 'ctabgan_study' in globals() and ctabgan_study is not None:
            best_trial = ctabgan_study.best_trial
            best_objective_score = best_trial.value
            trial_number = best_trial.number
            print(f"✅ Section 4.2 CTAB-GAN optimization completed successfully!")
            print(f"   • Best Trial: #{trial_number}")
        else:
            # Use fallback values when memory unavailable
            best_objective_score = 0.0
            trial_number = "loaded_from_csv"
            print(f"✅ Section 4.2 CTAB-GAN parameters loaded from CSV!")
            print(f"   • Best Trial: #{trial_number}")
        
        print(f"   • Best Objective Score: {best_objective_score:.4f}" if isinstance(best_objective_score, (int, float)) else f"   • Best Objective Score: {best_objective_score}")
        print(f"   • Best Parameters:")
        for param, value in best_params.items():
            print(f"     - {param}: {value}")
        
        # 5.2.2 Train Final CTAB-GAN Model using Section 5.1 Pattern
        print("🔧 Training final CTAB-GAN model using Section 5.1 proven pattern with optimized parameters...")
        
        try:
            # Use the exact same ModelFactory pattern that works in Section 5.1
            from src.models.model_factory import ModelFactory
            
            # Create CTAB-GAN model using the working pattern
            final_ctabgan_model = ModelFactory.create("ctabgan", random_state=42)
            
            # Apply the best parameters found in Section 4.2 optimization
            final_ctabgan_params = {
                'epochs': best_params.get('epochs', 300),
                'batch_size': best_params.get('batch_size', 512),
                'lr': best_params.get('lr', 2e-4),
                'betas': best_params.get('betas', (0.5, 0.9)),
                'l2scale': best_params.get('l2scale', 1e-5),
                'mixed_precision': best_params.get('mixed_precision', False),
                'test_ratio': best_params.get('test_ratio', 0.20),
                'verbose': best_params.get('verbose', True)
            }
            
            print("🔧 Training CTAB-GAN with optimal hyperparameters...")
            for param, value in final_ctabgan_params.items():
                print(f"   • Using {param}: {value}")
            
            # Train the model with best parameters
            final_ctabgan_model.train(data, **final_ctabgan_params)
            print("✅ CTAB-GAN training completed successfully!")
            
            # Generate synthetic data
            print("📊 Generating synthetic data for evaluation...")
            synthetic_ctabgan_final = final_ctabgan_model.generate(len(data))
            print(f"✅ Generated {len(synthetic_ctabgan_final)} synthetic samples")
            
            # Evaluate using enhanced objective function
            if 'enhanced_objective_function_v2' in globals():
                print("🎯 CTAB-GAN Classification Performance Analysis:")
                
                ctabgan_final_score, ctabgan_similarity, ctabgan_accuracy = enhanced_objective_function_v2(
                    real_data=data, 
                    synthetic_data=synthetic_ctabgan_final, 
                    target_column=TARGET_COLUMN
                )
                
                print(f"✅ CTAB-GAN Final Results:")
                print(f"   • Overall Score: {ctabgan_final_score:.4f}")
                print(f"   • Similarity Score: {ctabgan_similarity:.4f}")  
                print(f"   • Accuracy Score: {ctabgan_accuracy:.4f}")
                
                # Store results for Section 5.7 comparison
                ctabgan_final_results = {
                    'model_name': 'CTAB-GAN',
                    'objective_score': ctabgan_final_score,
                    'similarity_score': ctabgan_similarity,
                    'accuracy_score': ctabgan_accuracy,
                    'best_params': best_params,
                    'synthetic_data': synthetic_ctabgan_final
                }
                
            else:
                print("⚠️ Enhanced objective function not available - using basic evaluation")
                ctabgan_final_results = {
                    'model_name': 'CTAB-GAN',
                    'objective_score': best_objective_score,
                    'best_params': best_params,
                    'synthetic_data': synthetic_ctabgan_final
                }
                
        except Exception as e:
            print(f"❌ CTAB-GAN training failed: {str(e)}")
            synthetic_ctabgan_final = None
            ctabgan_final_results = {
                'model_name': 'CTAB-GAN',
                'objective_score': 0.0,
                'error': str(e)
            }
        
    else:
        print("❌ CTAB-GAN study results not found - Section 4.2 may not have completed successfully")
        print("    Please ensure Section 4.2 has been executed before running Section 5.2")
        synthetic_ctabgan_final = None
        ctabgan_final_score = 0.0
        ctabgan_final_results = {
            'model_name': 'CTAB-GAN',
            'objective_score': 0.0,
            'error': 'Section 4.2 not completed'
        }
        
except Exception as e:
    print(f"❌ Error in Section 5.2 CTAB-GAN evaluation: {e}")
    import traceback
    traceback.print_exc()
    synthetic_ctabgan_final = None
    ctabgan_final_score = 0.0
    ctabgan_final_results = {
        'model_name': 'CTAB-GAN',
        'objective_score': 0.0,
        'error': str(e)
    }

print("✅ Section 5.2 CTAB-GAN evaluation completed!")
print("=" * 60)

🏆 SECTION 5.2: BEST CTAB-GAN MODEL EVALUATION
📊 5.2.1 Retrieving best CTAB-GAN results from Section 4.2...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/liver-train/2025-12-16/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 12 parameters
[OK] Loaded CTAB-GAN: 3 parameters
[OK] Loaded CTAB-GAN+: 3 parameters
[OK] Loaded GANerAid: 3 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 7 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 12 parameters
   - ctabgan: 3 parameters
   - ctabganplus: 3 parameters
   - ganeraid: 3 parameters
   - copulagan: 6 parameters
   - tvae: 7 parameters
[OK] CTAB-GAN parameters loaded from CSV file
✅ Section 4.2 CTAB-GAN optimization completed successfully!
   • Best Trial: #2
   • Best Objective Score: 0.6196
   • Best Parameters:
     - epochs: 250
     - batch_size: 64
     - test_ratio: 0.25
🔧 Training final CTAB-GAN mo

100%|██████████| 250/250 [06:31<00:00,  1.57s/it]


Finished training in 393.2557723522186  seconds.
✅ CTAB-GAN training completed successfully!
📊 Generating synthetic data for evaluation...
✅ Generated 2500 synthetic samples
🎯 CTAB-GAN Classification Performance Analysis:
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5570
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7336
[CHART] Combined Score: 0.6276 (Similarity: 0.5570, Accuracy: 0.7336)
✅ CTAB-GAN Final Results:
   • Overall Score: 0.6276
   • Similarity Score: 0.5570
   • Accuracy Score: 0.7336
✅ Section 5.2 CTAB-GAN evaluation completed!


#### 5.1.3 Best CTAB-GAN+ Model Evaluation

In [39]:
# Code Chunk ID: CHUNK_5_1_3_A
# ============================================================================
# Section 5.3: Best CTAB-GAN+ Model Evaluation - FIXED IMPLEMENTATION
# ============================================================================
# Using Section 4.3 optimized hyperparameters with proven ModelFactory pattern

print("🏆 SECTION 5.3: BEST CTAB-GAN+ MODEL EVALUATION")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.3 CTAB-GAN+ optimization results
    if 'ctabganplus_study' in globals():
        best_trial = ctabganplus_study.best_trial
        best_params = best_trial.params
        best_objective_score = best_trial.value
        
        print(f"✅ Retrieved Section 4.3 CTAB-GAN+ optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(best_params)} hyperparameters")
        
        # Display best parameters
        print(f"\n📊 Best CTAB-GAN+ Hyperparameters:")
        print("-" * 40)
        for param, value in best_params.items():
            if isinstance(value, float):
                print(f"   • {param}: {value:.4f}")
            else:
                print(f"   • {param}: {value}")
                
    else:
        print("⚠️ CTAB-GAN+ optimization results not found - using fallback parameters")
        # Fallback CTAB-GAN+ parameters (basic working configuration)
        best_params = {
            'epochs': 100,
            'batch_size': 128,
            'lr_generator': 1e-4,
            'lr_discriminator': 2e-4,
            'beta_1': 0.5,
            'beta_2': 0.9,
            'lambda_gp': 10,
            'pac': 1
        }
        best_objective_score = None
        print(f"   Using fallback parameters: {best_params}")

    # Step 2: Create CTAB-GAN+ model using proven ModelFactory pattern (SAME AS SECTION 5.2)
    print(f"\n🏗️ Creating CTAB-GAN+ model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    
    # CRITICAL FIX: Use the exact same ModelFactory pattern that works in Section 5.1 & 5.2
    final_ctabganplus_model = ModelFactory.create("ctabganplus", random_state=42)
    print(f"✅ CTAB-GAN+ model created successfully")
    
    # Step 3: Train using the correct method name: .train() (NOT .fit())
    print(f"\n🚀 Training CTAB-GAN+ model with optimized hyperparameters...")
    print(f"   • Data shape: {data.shape}")
    print(f"   • Target column: '{TARGET_COLUMN}'")
    print(f"   • Training with Section 4.3 parameters")
    
    # Store final parameters for results tracking
    final_ctabganplus_params = best_params.copy()
    
    # CRITICAL FIX: Train using .train() method (proven pattern from Sections 5.1 & 5.2)
    final_ctabganplus_model.train(data, **final_ctabganplus_params)
    print(f"✅ CTAB-GAN+ model training completed successfully!")
    
    # Step 4: Generate synthetic data using the correct method: .generate()
    print(f"\n📊 Generating synthetic data for evaluation...")
    synthetic_ctabganplus_final = final_ctabganplus_model.generate(len(data))
    print(f"✅ Synthetic data generated successfully!")
    print(f"   • Synthetic data shape: {synthetic_ctabganplus_final.shape}")
    print(f"   • Columns match: {list(synthetic_ctabganplus_final.columns) == list(data.columns)}")
    
    # Step 5: Quick evaluation using enhanced objective function (NO IMPORT - function in globals)
    if 'enhanced_objective_function_v2' in globals():
        ctabganplus_final_score, ctabganplus_similarity, ctabganplus_accuracy = enhanced_objective_function_v2(
            real_data=data, 
            synthetic_data=synthetic_ctabganplus_final, 
            target_column=TARGET_COLUMN
        )
        
        print(f"\n📊 CTAB-GAN+ Enhanced Objective Function v2 Results:")
        print(f"   • Final Combined Score: {ctabganplus_final_score:.4f}")
        print(f"   • Statistical Similarity (60%): {ctabganplus_similarity:.4f}")
        print(f"   • Classification Accuracy (40%): {ctabganplus_accuracy:.4f}")
    else:
        print("⚠️ Enhanced objective function not available - using basic metrics")
        ctabganplus_final_score = 0.5  # Fallback score
        ctabganplus_similarity = 0.5
        ctabganplus_accuracy = 0.5
    
    # Store results for Section 5.7 comparative analysis
    ctabganplus_final_results = {
        'model_name': 'CTAB-GAN+',
        'objective_score': ctabganplus_final_score,
        'similarity_score': ctabganplus_similarity,
        'accuracy_score': ctabganplus_accuracy,
        'final_combined_score': ctabganplus_final_score,
        'sections_completed': ['5.3.1'],
        'evaluation_method': 'section_5_1_pattern',
        'section_4_optimization': best_objective_score is not None,
        'best_section_4_score': best_objective_score
    }
    
    print(f"\n✅ SECTION 5.3 COMPLETED SUCCESSFULLY!")
    print(f"🎯 CTAB-GAN+ evaluation completed using Section 4.3 optimized parameters")
    print(f"📊 Results ready for Section 5.7 comparative analysis")
    print("-" * 80)

except Exception as e:
    print(f"❌ CTAB-GAN+ evaluation failed: {str(e)}")
    import traceback
    traceback.print_exc()
    # Set fallback for subsequent sections
    synthetic_ctabganplus_final = None
    ctabganplus_final_results = {'error': str(e), 'evaluation_failed': True}

🏆 SECTION 5.3: BEST CTAB-GAN+ MODEL EVALUATION
✅ Retrieved Section 4.3 CTAB-GAN+ optimization results
   • Best Trial: #2
   • Best Objective Score: 0.6384
   • Parameters: 3 hyperparameters

📊 Best CTAB-GAN+ Hyperparameters:
----------------------------------------
   • epochs: 400
   • batch_size: 256
   • test_ratio: 0.2000

🏗️ Creating CTAB-GAN+ model using ModelFactory...
✅ CTAB-GAN+ model created successfully

🚀 Training CTAB-GAN+ model with optimized hyperparameters...
   • Data shape: (2500, 11)
   • Target column: 'Result'
   • Training with Section 4.3 parameters


100%|██████████| 400/400 [39:24<00:00,  5.91s/it]


Finished training in 2365.5050461292267  seconds.
✅ CTAB-GAN+ model training completed successfully!

📊 Generating synthetic data for evaluation...
✅ Synthetic data generated successfully!
   • Synthetic data shape: (2500, 11)
   • Columns match: True
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.4883
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7222
[CHART] Combined Score: 0.5819 (Similarity: 0.4883, Accuracy: 0.7222)

📊 CTAB-GAN+ Enhanced Objective Function v2 Results:
   • Final Combined Score: 0.5819
   • Statistical Similarity (60%): 0.4883
   • Classification Accuracy (40%): 0.7222

✅ SECTION 5.3 COMPLETED SUCCESSFULLY!
🎯 CTAB-GAN+ evaluation completed using Section 4.3 optimized parameters
📊 Results ready for Section 5.7 comparative analysis
--------------------------------------------------------------------------------


#### Section 5.1.4 BEST GANerAid MODEL

In [40]:
# Code Chunk ID: CHUNK_5_1_4_A
# ============================================================================
# Section 5.4.1: Best GANerAid Model Training
# ============================================================================
# Using Section 4.4 optimized hyperparameters with proven ModelFactory pattern

print("🏆 SECTION 5.4.1: BEST GANerAid MODEL TRAINING")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.4 GANerAid optimization results
    if 'ganeraid_study' in globals():
        best_trial = ganeraid_study.best_trial
        final_ganeraid_params = best_trial.params
        best_objective_score = best_trial.value
        
        print(f"✅ Retrieved Section 4.4 GANerAid optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(final_ganeraid_params)} hyperparameters")
        
    else:
        print("⚠️ GANerAid optimization results not found - using fallback parameters")
        # Fallback GANerAid parameters
        final_ganeraid_params = {
            'epochs': 100,
            'batch_size': 128,
            'learning_rate': 1e-4
        }
        best_objective_score = None

    # Step 2: Create GANerAid model using proven ModelFactory pattern
    print(f"\n🏗️ Creating GANerAid model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    
    final_ganeraid_model = ModelFactory.create("ganeraid", random_state=42)
    print(f"✅ GANerAid model created successfully")
    
    # Step 3: Train using .train() method (NOT .fit())
    print(f"\n🚀 Training GANerAid model with optimized hyperparameters...")
    final_ganeraid_model.train(data, **final_ganeraid_params)
    print(f"✅ GANerAid model training completed successfully!")
    
    # Step 4: Generate synthetic data
    synthetic_ganeraid_final = final_ganeraid_model.generate(len(data))
    print(f"✅ GANerAid synthetic data generated: {synthetic_ganeraid_final.shape}")
    
    # Step 5: Quick evaluation using enhanced objective function (NO IMPORT - function in globals)
    if 'enhanced_objective_function_v2' in globals():
        ganeraid_final_score, ganeraid_similarity, ganeraid_accuracy = enhanced_objective_function_v2(
            real_data=data, synthetic_data=synthetic_ganeraid_final, target_column=TARGET_COLUMN
        )
        
        print(f"\n📊 GANerAid Enhanced Objective Function v2 Results:")
        print(f"   • Final Combined Score: {ganeraid_final_score:.4f}")
        print(f"   • Statistical Similarity (60%): {ganeraid_similarity:.4f}")
        print(f"   • Classification Accuracy (40%): {ganeraid_accuracy:.4f}")
    else:
        print("⚠️ Enhanced objective function not available - using basic metrics")
        ganeraid_final_score = 0.5  # Fallback score
        ganeraid_similarity = 0.5
        ganeraid_accuracy = 0.5
    
    # Store results
    ganeraid_final_results = {
        'model_name': 'GANerAid',
        'objective_score': ganeraid_final_score,
        'similarity_score': ganeraid_similarity,
        'accuracy_score': ganeraid_accuracy,
        'final_combined_score': ganeraid_final_score,
        'sections_completed': ['5.4.1'],
        'evaluation_method': 'section_5_1_pattern',
        'section_4_optimization': best_objective_score is not None,
        'best_section_4_score': best_objective_score,
        'optimized_params': final_ganeraid_params
    }
    
    print(f"\n✅ SECTION 5.4.1 - GANerAid MODEL TRAINING COMPLETED!")
    print("-" * 80)

except Exception as e:
    print(f"❌ GANerAid training failed: {str(e)}")
    import traceback
    traceback.print_exc()
    synthetic_ganeraid_final = None
    ganeraid_final_results = {'error': str(e), 'training_failed': True}

WARNING	src.models.implementations.ganeraid_model:ganeraid_model.py:train()- Batch size compatibility: batch_size=128 % nr_of_rows=25 = 3


🏆 SECTION 5.4.1: BEST GANerAid MODEL TRAINING
✅ Retrieved Section 4.4 GANerAid optimization results
   • Best Trial: #4
   • Best Objective Score: 0.6006
   • Parameters: 3 hyperparameters

🏗️ Creating GANerAid model using ModelFactory...
✅ GANerAid model created successfully

🚀 Training GANerAid model with optimized hyperparameters...
Initialized gan with the following parameters: 
lr_d = 0.0005
lr_g = 0.0005
hidden_feature_space = 200
batch_size = 128
nr_of_rows = 25
binary_noise = 0.2
Start training of gan for 1300 epochs


100%|██████████| 1300/1300 [05:20<00:00,  4.06it/s, loss=d error: 0.8342424631118774 --- g error 1.978318214416504]  


✅ GANerAid model training completed successfully!
Generating 2500 samples
✅ GANerAid synthetic data generated: (2500, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.4864
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7096
[CHART] Combined Score: 0.5757 (Similarity: 0.4864, Accuracy: 0.7096)

📊 GANerAid Enhanced Objective Function v2 Results:
   • Final Combined Score: 0.5757
   • Statistical Similarity (60%): 0.4864
   • Classification Accuracy (40%): 0.7096

✅ SECTION 5.4.1 - GANerAid MODEL TRAINING COMPLETED!
--------------------------------------------------------------------------------


#### 5.1.5: Best CopulaGAN Model

In [41]:
# Code Chunk ID: CHUNK_5_1_5_A
# ============================================================================
# Section 5.5: Best CopulaGAN Model Evaluation
# ============================================================================
# Using Section 4.5 optimized hyperparameters with proven ModelFactory pattern

print("🎯 ============================================================================")
print("🎯 Section 5.5: CopulaGAN Final Model Training & Evaluation")
print("🎯 ============================================================================")

try:
    # Load all best parameters using setup.py function
    param_data = load_best_parameters_from_csv(
        section_number=4,
        dataset_identifier=DATASET_IDENTIFIER,
        fallback_to_memory=True,
        scope=globals()
    )
    
    # Extract CopulaGAN parameters specifically
    loaded_copulagan_params = param_data['parameters'].get('copulagan', None)
    
    if loaded_copulagan_params:
        print(f"📋 Loaded CopulaGAN parameters from {param_data.get('source', 'unknown')}:")
        for param, value in loaded_copulagan_params.items():
            print(f"   • {param}: {value}")
        best_params = loaded_copulagan_params.copy()
        param_source = param_data.get('source', 'CSV')
    else:
        print("⚠️ CopulaGAN optimization results not found - using fallback parameters")
        best_params = {'epochs': 50, 'batch_size': 64, 'lr': 0.0002}
        param_source = 'fallback'
    
    print(f"\n🔧 Preprocessing data for CopulaGAN...")
    data_preprocessed = data.copy()
    print(f"   ✅ Data preprocessing completed: {data_preprocessed.shape}")
    print(f"   • Missing values: {data_preprocessed.isnull().sum().sum()}")
    
    # Show data types
    dtype_counts = data_preprocessed.dtypes.value_counts()
    print(f"   • Data types: {dict(dtype_counts)}")
    
    print(f"\n🏗️ Creating CopulaGAN model using ModelFactory...")
    final_copulagan_model = ModelFactory.create("copulagan", random_state=42)
    print("✅ CopulaGAN model created successfully")
    
    print(f"\n🚀 Training CopulaGAN model with optimized hyperparameters...")
    print(f"   • Using parameters: {best_params}")
    print(f"   • Using ALL parameters from Section 4.5: {best_params}")
    
    # Train the model with best parameters - using .train() method like other Section 5 models
    final_copulagan_model.train(data_preprocessed, **best_params)
    
    print("✅ CopulaGAN model training completed successfully")
    
    # Generate synthetic data
    print(f"\n🎲 Generating {len(data_preprocessed)} synthetic samples...")
    synthetic_copulagan_final = final_copulagan_model.generate(len(data_preprocessed))
    print(f"   ✅ Generated synthetic data: {synthetic_copulagan_final.shape}")
    
    # Comprehensive evaluation with enhanced objective function - using correct parameters
    print(f"\n📊 Comprehensive model evaluation...")
    copulagan_final_score, copulagan_similarity, copulagan_accuracy = enhanced_objective_function_v2(
        real_data=data_preprocessed,
        synthetic_data=synthetic_copulagan_final,
        target_column=TARGET_COLUMN
    )
    
    print(f"\n🎯 === CopulaGAN Final Results (Section 5.5) ===")
    print(f"📈 Combined Score: {copulagan_final_score:.4f}")
    print(f"📊 Similarity Score: {copulagan_similarity:.4f}")
    print(f"🎯 Accuracy Score: {copulagan_accuracy:.4f}")
    print(f"⚙️ Best Parameters: {best_params}")
    print(f"📁 Parameter Source: {param_source}")
    print(f"=====================================")
    
    # Store results for Section 5.2 batch processing
    copulagan_final_results = {
        'model_name': 'CopulaGAN',
        'combined_score': copulagan_final_score,
        'similarity_score': copulagan_similarity,
        'accuracy_score': copulagan_accuracy,
        'best_params': best_params,
        'parameter_source': param_data.get('source', 'memory'),
        'synthetic_data': synthetic_copulagan_final
    }
    
    print(f"💾 Results stored for Section 5.2 batch processing")
    
except Exception as e:
    error_msg = str(e)
    print(f"❌ CopulaGAN model creation/training failed: {error_msg}")
    print(f"   This may be due to CopulaGAN compatibility issues")
    
    # Fallback handling - store minimal results
    copulagan_final_results = {
        'model_name': 'CopulaGAN',
        'combined_score': 0.0,
        'similarity_score': 0.0,
        'accuracy_score': 0.0,
        'best_params': {'epochs': 50, 'batch_size': 64, 'lr': 0.0002},
        'parameter_source': 'fallback',
        'error': error_msg,
        'synthetic_data': None
    }
    
    print(f"💾 Fallback results stored for Section 5.2 batch processing")

🎯 ============================================================================
🎯 Section 5.5: CopulaGAN Final Model Training & Evaluation
🎯 ============================================================================
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/liver-train/2025-12-16/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 12 parameters
[OK] Loaded CTAB-GAN: 3 parameters
[OK] Loaded CTAB-GAN+: 3 parameters
[OK] Loaded GANerAid: 3 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 7 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 6
   - ctgan: 12 parameters
   - ctabgan: 3 parameters
   - ctabganplus: 3 parameters
   - ganeraid: 3 parameters
   - copulagan: 6 parameters
   - tvae: 7 parameters
📋 Loaded CopulaGAN parameters from CSV file:
   • epochs: 400
   • batch_size: 500
   • generator_lr: 0.0028960879454961222
   • discriminator_lr: 0.0083727074751052

#### 5.1.6: Best TVAE Model Evaluation 

In [42]:
# Code Chunk ID: CHUNK_5_1_6_A
# ============================================================================
# Section 5.6.1: Best TVAE Model Training
# ============================================================================
# Using Section 4.6 optimized hyperparameters with proven ModelFactory pattern

print("🏆 SECTION 5.6.1: BEST TVAE MODEL TRAINING")
print("=" * 80)

try:
    # Step 1: Retrieve Section 4.6 TVAE optimization results
    if 'tvae_study' in globals():
        best_trial = tvae_study.best_trial
        final_tvae_params = best_trial.params
        best_objective_score = best_trial.value
        
        print(f"✅ Retrieved Section 4.6 TVAE optimization results")
        print(f"   • Best Trial: #{best_trial.number}")
        print(f"   • Best Objective Score: {best_objective_score:.4f}")
        print(f"   • Parameters: {len(final_tvae_params)} hyperparameters")
        
    else:
        print("⚠️ TVAE optimization results not found - using fallback parameters")
        # Fallback TVAE parameters
        final_tvae_params = {
            'epochs': 100,
            'batch_size': 128,
            'lr': 1e-4,
            'compress_dims': [128, 64],
            'decompress_dims': [64, 128]
        }
        best_objective_score = None

    # Step 2: Create TVAE model using proven ModelFactory pattern
    print(f"\n🏗️ Creating TVAE model using ModelFactory...")
    from src.models.model_factory import ModelFactory
    
    final_tvae_model = ModelFactory.create("tvae", random_state=42)
    print(f"✅ TVAE model created successfully")
    
    # Step 3: Train using .train() method (NOT .fit())
    print(f"\n🚀 Training TVAE model with optimized hyperparameters...")
    final_tvae_model.train(data, **final_tvae_params)
    print(f"✅ TVAE model training completed successfully!")
    
    # Step 4: Generate synthetic data
    synthetic_tvae_final = final_tvae_model.generate(len(data))
    print(f"✅ TVAE synthetic data generated: {synthetic_tvae_final.shape}")
    
    # Step 5: Quick evaluation using enhanced objective function (NO IMPORT - function in globals)
    if 'enhanced_objective_function_v2' in globals():
        tvae_final_score, tvae_similarity, tvae_accuracy = enhanced_objective_function_v2(
            real_data=data, synthetic_data=synthetic_tvae_final, target_column=TARGET_COLUMN
        )
        
        print(f"\n📊 TVAE Enhanced Objective Function v2 Results:")
        print(f"   • Final Combined Score: {tvae_final_score:.4f}")
        print(f"   • Statistical Similarity (60%): {tvae_similarity:.4f}")
        print(f"   • Classification Accuracy (40%): {tvae_accuracy:.4f}")
    else:
        print("⚠️ Enhanced objective function not available - using basic metrics")
        tvae_final_score = 0.5  # Fallback score
        tvae_similarity = 0.5
        tvae_accuracy = 0.5
    
    # Store results
    tvae_final_results = {
        'model_name': 'TVAE',
        'objective_score': tvae_final_score,
        'similarity_score': tvae_similarity,
        'accuracy_score': tvae_accuracy,
        'final_combined_score': tvae_final_score,
        'sections_completed': ['5.6.1'],
        'evaluation_method': 'section_5_1_pattern',
        'section_4_optimization': best_objective_score is not None,
        'best_section_4_score': best_objective_score,
        'optimized_params': final_tvae_params
    }
    
    print(f"\n✅ SECTION 5.6.1 - TVAE MODEL TRAINING COMPLETED!")
    print("-" * 80)

except Exception as e:
    print(f"❌ TVAE training failed: {str(e)}")
    import traceback
    traceback.print_exc()
    synthetic_tvae_final = None
    tvae_final_results = {'error': str(e), 'training_failed': True}

🏆 SECTION 5.6.1: BEST TVAE MODEL TRAINING
✅ Retrieved Section 4.6 TVAE optimization results
   • Best Trial: #3
   • Best Objective Score: 0.6468
   • Parameters: 7 hyperparameters

🏗️ Creating TVAE model using ModelFactory...
✅ TVAE model created successfully

🚀 Training TVAE model with optimized hyperparameters...
✅ TVAE model training completed successfully!
✅ TVAE synthetic data generated: (2500, 11)
[TARGET] Enhanced objective function using target column: 'Result'
[OK] Similarity Analysis: 11/11 valid metrics, Average: 0.5527
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7464
[CHART] Combined Score: 0.6302 (Similarity: 0.5527, Accuracy: 0.7464)

📊 TVAE Enhanced Objective Function v2 Results:
   • Final Combined Score: 0.6302
   • Statistical Similarity (60%): 0.5527
   • Classification Accuracy (40%): 0.7464

✅ SECTION 5.6.1 - TVAE MODEL TRAINING COMPLETED!
--------------------------------------------------------------------------------


### 5.2 Batch Process

This section outputs figures and graphics from models run in 5.1. The next chunk will output results for whatever subsections of 5.1 were run. I.e., if there's need to debug one method, there's no need to run all subsections of 5.1.

In [43]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# Following CHUNK_018 pattern with comprehensive file export to Section-5 directory
# ============================================================================

print("🔍 SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)
print("📋 Evaluating all available optimized models from Section 5.1.x")
print("📁 Exporting all tables and analysis to Section-5 directory")
print("🔄 Following Section 3 comprehensive evaluation pattern")
print()

# Ensure setup module function is available
from setup import evaluate_section5_optimized_models

# Use Section 5 batch evaluation function from setup.py
# Following exact same pattern as CHUNK_018 (Section 3) - comprehensive file export!
try:
    # Run batch evaluation with file export for all optimized models
    section5_batch_results = evaluate_section5_optimized_models(
        section_number=5,
        scope=globals(),  # Pass notebook scope to access synthetic data variables
        target_column=TARGET_COLUMN
    )
    
    print("\n" + "="*80)
    print("✅ SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
    print("="*80)
    print(f"📊 Models processed: {section5_batch_results['models_processed']}")
    print(f"📁 Results exported to: {section5_batch_results['results_dir']}")
    
    # Show summary of all evaluations
    if 'evaluation_summaries' in section5_batch_results:
        print("\n📋 EVALUATION SUMMARIES:")
        print("-" * 40)
        for model_name, summary in section5_batch_results['evaluation_summaries'].items():
            print(f"🤖 {model_name}:")
            print(f"   📊 Synthetic samples: {summary.get('synthetic_samples', 'N/A')}")
            print(f"   📈 Overall score: {summary.get('overall_score', 'N/A')}")
            
    print("\n" + "="*80)
            
except Exception as e:
    print(f"❌ Section 5.2 batch evaluation failed: {e}")
    print(f"🔍 Error details: {type(e).__name__}")
    print()
    print("⚠️  Check that Section 5.1.x models completed successfully")

print("\n📈 Section 5.2 optimized model batch evaluation complete!")
print("🏁 Ready for final model comparison and production deployment!")

🔍 SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
📋 Evaluating all available optimized models from Section 5.1.x
📁 Exporting all tables and analysis to Section-5 directory
🔄 Following Section 3 comprehensive evaluation pattern

[SEARCH] UNIFIED BATCH EVALUATION - SECTION 5
[INFO] Dataset: liver-train
[INFO] Target column: Result
[INFO] Variable pattern: final
[INFO] Found 6 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] GANerAid
   [OK] CopulaGAN
   [OK] TVAE

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results\liver-train\2025-12-16\Section-5\CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.879

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0

WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
WARNING	matplotlib.legend:legend.py:_parse_legend_args()- No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


[SUCCESS] ROC curves saved: results/liver-train/2025-12-16/Section-5\trts_roc_curves.png

GENERATING PRECISION-RECALL CURVES
[WARNING] Could not generate PR curve for CTGAN TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for CTABGAN TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for CTABGANPLUS TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for GANerAid TRTR: y_true takes value in {1, 2} and pos_label is not specified: either make y_true take value in {0, 1} or {-1, 1} or pass pos_label explicitly.
[WARNING] Could not generate PR curve for CopulaGAN TRTR: y_true takes value in 

In [44]:
# Generate Section 5 README documentation
from src.utils.documentation import create_section5_readme

create_section5_readme(
    results_path=results_path,
    dataset_id=DATASET_IDENTIFIER,
    timestamp=SESSION_TIMESTAMP
)


[DOCS] Created: Section-5/README.md


'results/liver-train/2025-12-16/Section-2/README.md'